# 特徵工程＆分訓練測試資料

In [1]:
# 匯入資料
import pandas as pd

# 路徑
file_transaction = "/work/claire901114/初賽資料/acct_transaction.csv"
file_alert = "/work/claire901114/初賽資料/acct_alert.csv"
file_predict = "/work/claire901114/初賽資料/acct_predict.csv"

# 讀檔
df_transaction = pd.read_csv(file_transaction)
df_alert = pd.read_csv(file_alert)
df_predict = pd.read_csv(file_predict)

# 檢查資料結構
print("📂 acct_transaction.csv")
print(df_transaction.info())
print(df_transaction.head(), "\n")

print("📂 acct_alert.csv")
print(df_alert.info())
print(df_alert.head(), "\n")

print("📂 acct_predict.csv")
print(df_predict.info())
print(df_predict.head(), "\n")

📂 acct_transaction.csv
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4435890 entries, 0 to 4435889
Data columns (total 10 columns):
 #   Column          Dtype  
---  ------          -----  
 0   from_acct       object 
 1   from_acct_type  int64  
 2   to_acct         object 
 3   to_acct_type    int64  
 4   is_self_txn     object 
 5   txn_amt         float64
 6   txn_date        int64  
 7   txn_time        object 
 8   currency_type   object 
 9   channel_type    object 
dtypes: float64(1), int64(3), object(6)
memory usage: 338.4+ MB
None
                                           from_acct  from_acct_type  \
0  be6fdd2d0f9aa02b0b09436fb137654942e3346e16ab43...               1   
1  18f3d0e79217f8bc8b4cb485f9f80a884771b846de652f...               1   
2  302f3911cbf56bf9b5ad209a4b045a82380f98d92604c1...               1   
3  5a4809796865b1526f46e5dda6a35c1a4def3cbe969cc8...               1   
4  7f84214987bdee16ffbaf3d70824e6385ce80e032a24c5...               1   

              

In [2]:
# 讀取帳戶名單
train_accts = set(df_alert["acct"].unique())
test_accts = set(df_predict["acct"].unique())

In [3]:
# 標記帳戶來源
df_transaction["from_in_train"] = df_transaction["from_acct"].isin(train_accts)
df_transaction["to_in_train"]   = df_transaction["to_acct"].isin(train_accts)
df_transaction["from_in_test"]  = df_transaction["from_acct"].isin(test_accts)
df_transaction["to_in_test"]    = df_transaction["to_acct"].isin(test_accts)

In [10]:
import numpy as np
from catboost import CatBoostClassifier
from tqdm.auto import tqdm
tqdm.pandas()

# 檔案
file_transaction = "/work/claire901114/初賽資料/acct_transaction.csv"
file_alert = "/work/claire901114/初賽資料/acct_alert.csv"
file_predict = "/work/claire901114/初賽資料/acct_predict.csv"

# 定義數據類型: 將字串標籤降轉為 category，數值欄位降轉為 float/int
dtypes_txn = {
    "from_acct": "category", "from_acct_type": "category",
    "to_acct": "category", "to_acct_type": "category",
    "is_self_txn": "category", "txn_amt": "float32",
    "txn_date": "int32", "txn_time": "string",
    "currency_type": "category", "channel_type": "category"
}

# ====== 讀取檔案 ======
df_transaction = pd.read_csv(file_transaction, dtype=dtypes_txn)
df_alert = pd.read_csv(file_alert, dtype={"acct": "category", "event_date": "int32"})
df_predict = pd.read_csv(file_predict, dtype={"acct": "category"})


# 訓練集/預測集帳號名單
train_accts = set(df_alert["acct"]) # 正樣本 (Label=1)
test_accts = set(df_predict["acct"])
all_target_accts = train_accts.union(test_accts)
# 初始化警示帳戶
alert_accts_set = set(df_alert["acct"]) 

print("✅ 訓練集正樣本帳戶數:", len(train_accts))
print("✅ 測試帳戶數:", len(test_accts))
print(f"✅ 已知警示帳戶數: {len(alert_accts_set)}")


# =====================================================
# 0. Helper Functions(統計指標、時間維度、黑名單社交圖譜特徵計算)
# =====================================================

# 金額變異係數: 詐騙洗錢（如車手分流、人頭戶提領）可能常呈現固定金額或規律比例轉帳之特徵
def calculate_cov(x):
    if len(x) <= 1: return 0.0
    mean_val = x.mean()
    # 返回 float 型態
    return 0.0 if mean_val == 0 else x.std() / mean_val

# 時間區間分類 詐騙集中於深夜與清晨 (22:00 - 06:00) 進行頻繁跨行轉帳
def time_period(h):
    if np.isnan(h): return "UNK"
    h = int(h)
    if 0 <= h < 6 or h >= 22: return "night"
    elif 6 <= h < 12: return "morning"
    elif 12 <= h < 18: return "afternoon"
    else: return "evening"

# 警示對手帳戶特徵
def calculate_alert_partner_features(df, alert_set):
    df_temp = df.copy()
    
    # 特徵 1: 匯款給已知警示帳戶
    df_temp["to_is_alert"] = df_temp["to_acct"].isin(alert_set)
    agg_out_to_alert = df_temp.loc[df_temp["to_is_alert"] == True].groupby("from_acct", observed=True).agg(
        alert_partner_out_cnt=("txn_amt", "count"), 
        alert_partner_out_sum=("txn_amt", "sum")   
    ).rename_axis("acct").reset_index()

    # 特徵 2: 從已知警示帳戶收款
    df_temp["from_is_alert"] = df_temp["from_acct"].isin(alert_set)
    agg_in_from_alert = df_temp.loc[df_temp["from_is_alert"] == True].groupby("to_acct", observed=True).agg(
        alert_partner_in_cnt=("txn_amt", "count"), 
        alert_partner_in_sum=("txn_amt", "sum")   
    ).rename_axis("acct").reset_index()

    return agg_out_to_alert, agg_in_from_alert

# =====================================================
# 1. 資料讀取與清理與初始化
# =====================================================

print("--- 步驟 1: 資料讀取與清理與初始化 ---")

# 填補分類型缺失值
df_transaction['is_self_txn'] = df_transaction['is_self_txn'].fillna('UNK')

# 時間解析，避免非法時間格式造成程式中斷
time_dt = pd.to_datetime(
    df_transaction["txn_time"], format="%H:%M:%S", errors="coerce"
)
df_transaction['txn_hour'] = time_dt.dt.hour.astype("float32")
df_transaction['txn_minute'] = time_dt.dt.minute.astype("float32")

# 計算全域絕對時間軸（分鐘級），作為以後交易速率/時間間隔的計算依據
df_transaction['txn_minute_of_day'] = df_transaction['txn_date'] * 24 * 60 + df_transaction['txn_hour'] * 60 + df_transaction['txn_minute']

# 應用時間分段，並指定為 category 型別以利樹模型直接讀取
df_transaction['time_period'] = df_transaction['txn_hour'].apply(time_period).astype("category")

# 新增收款帳戶全局頻率計算 詐騙集團常頻繁購買、啟用全新的人頭帳戶。
#          「低度連結、突發性交易」的孤立節點特徵。
to_acct_freq = df_transaction['to_acct'].value_counts()
# 稀有門檻：在所有交易中出現次數小於 10 次的帳戶
RARE_THRESHOLD = 10
rare_accts = set(to_acct_freq[to_acct_freq < RARE_THRESHOLD].index)
print(f"✅ 稀有收款帳戶數量 (出現 < {RARE_THRESHOLD} 次): {len(rare_accts):,} 個")


print("✅ 資料讀取與清理完成。")


# =====================================================
# 1.5. 嚴格時序性切割 防止資料洩漏
# =====================================================

SPLIT_DAY = df_alert['event_date'].max()

MAX_TXN_DATE = df_transaction['txn_date'].max()
print(f"✅ 資料總日期範圍: 1 ~ {MAX_TXN_DATE} 天")
print(f"✅ 訓練集特徵截止日 (SPLIT_DAY): {SPLIT_DAY} 天")

# 訓練集特徵：只使用 SPLIT_DAY 之前的交易
df_transaction_train = df_transaction.loc[df_transaction['txn_date'] < SPLIT_DAY].copy()

# 測試集特徵：使用完整的交易資料來計算特徵
df_transaction_test = df_transaction.copy() 

print(f"✅ 訓練集交易筆數 (時間受限): {len(df_transaction_train):,} 筆")

# 訓練母體抽樣
# 訓練集候選帳戶：在 SPLIT_DAY 前曾參與交易的玉山帳戶 (From或To)
esun_from_train = df_transaction_train.loc[df_transaction_train["from_acct_type"]=="01", "from_acct"]
esun_to_train = df_transaction_train.loc[df_transaction_train["to_acct_type"]=="01", "to_acct"]
# 包含所有潛在正負樣本
train_candidate_accts = set(pd.concat([esun_from_train, esun_to_train]))
print(f"✅ 訓練集候選帳戶總數 (含正負樣本): {len(train_candidate_accts):,}")


✅ 訓練集正樣本帳戶數: 1004
✅ 測試帳戶數: 4780
✅ 已知警示帳戶數: 1004
--- 步驟 1: 資料讀取與清理與初始化 ---
✅ 稀有收款帳戶數量 (出現 < 10 次): 1,105,628 個
✅ 資料讀取與清理完成。
✅ 資料總日期範圍: 1 ~ 121 天
✅ 訓練集特徵截止日 (SPLIT_DAY): 121 天
✅ 訓練集交易筆數 (時間受限): 4,378,875 筆
✅ 訓練集候選帳戶總數 (含正負樣本): 332,208


In [11]:
# =====================================================
# 2. 特徵工程 (Features Engineering) - 封裝為函數
# =====================================================

# 1. 時序滑動視窗特徵 人頭帳戶在觸發警示前，可能有短期內（近7天）資金週轉率急遽飆升的異常特徵。
def create_features(df_txn, alert_set, target_accts): 
    
    print(" [1/9] 計算 7 天內交易速度特徵...")
    # 計算該 period 的最後一天，用於定義 "最後 7 天"
    max_date = df_txn['txn_date'].max()
    df_txn_7d = df_txn[df_txn['txn_date'] > max_date - 7].copy()
    
    # 7天內匯款方聚合：量化短期內的資金流出強度
    agg_out_7d = df_txn_7d.groupby("from_acct", observed=True).agg(
        out_sum_7d=("txn_amt", "sum"),
        out_count_7d=("txn_amt", "count"),
    ).rename_axis("acct").reset_index()
    
    # 2. 交易節奏與時間間隔特徵：計算連續交易的時間差（分鐘級別），或短時間內密集人頭戶提領（極低標準差）的異常行為。
    print(" [2/9] 計算交易節奏 (間隔) 特徵 (Progress Bar)...")
    # 先排序
    df_txn_paced = df_txn.sort_values(['from_acct', 'txn_minute_of_day'])
    # 每一筆交易與前一筆交易的時間間隔 (分鐘)
    df_txn_paced['txn_gap'] = df_txn_paced.groupby('from_acct')['txn_minute_of_day'].diff()
    # 群組內統計聚合：提取最短交易間隔與時間變異標準差
    agg_pacing_series = df_txn_paced.groupby('from_acct', observed=True)['txn_gap'].progress_apply(
        lambda x: pd.Series({'txn_gap_min': x.min(), 'txn_gap_std': x.std()})
    )
    agg_pacing = agg_pacing_series.unstack().rename_axis('acct').reset_index()


    # 2.5 新增：單小時最大交易筆數 (集中度指標)
    print(" [3/9] 計算單小時最大筆數特徵...")
    # 計算每個帳戶在單一小時內發生的最大交易次數
    hourly_max_count = df_txn.groupby(['from_acct', 'txn_hour']).size().reset_index(name='Hourly_Count')
    max_hourly_agg = hourly_max_count.groupby('from_acct')['Hourly_Count'].max().reset_index(name='Max_Hourly_Count').rename(columns={'from_acct': 'acct'})


    # 3. 基礎聚合特徵 建利多維度金流基線描述（包含總量、平均值、標準差與波動係數）
    print(" [4/9] 計算基礎聚合特徵 (金額/筆數/極值/標準差)...")
    # 匯款方基礎特徵群
    agg_out_base = df_txn.groupby("from_acct", observed=True).agg(
        out_sum=("txn_amt", "sum"), out_mean=("txn_amt", "mean"), out_std=("txn_amt", "std"), 
        out_max=("txn_amt", "max"), out_count=("txn_amt", "count")
    ).rename_axis("acct").reset_index()
    # 收款方基礎特徵群
    agg_in_base = df_txn.groupby("to_acct", observed=True).agg(
        in_sum=("txn_amt", "sum"), in_mean=("txn_amt", "mean"), in_std=("txn_amt", "std"), 
        in_max=("txn_amt", "max"), in_count=("txn_amt", "count")
    ).rename_axis("acct").reset_index()
    
    print(" [5/9] 計算變異係數 (CoV) 特徵 (Progress Bar)...")
    out_cov_series = df_txn.groupby("from_acct", observed=True)['txn_amt'].progress_apply(calculate_cov)
    out_cov_df = out_cov_series.rename("out_cov").reset_index().rename(columns={'from_acct': 'acct'})
    
    in_cov_series = df_txn.groupby("to_acct", observed=True)['txn_amt'].progress_apply(calculate_cov)
    in_cov_df = in_cov_series.rename("in_cov").reset_index().rename(columns={'to_acct': 'acct'})

    # 彙整結果
    agg_out = agg_out_base.merge(out_cov_df, on='acct', how='left')
    numeric_cols_out = agg_out.columns.drop('acct')
    agg_out[numeric_cols_out] = agg_out[numeric_cols_out].fillna(0)
    
    agg_in = agg_in_base.merge(in_cov_df, on='acct', how='left')
    numeric_cols_in = agg_in.columns.drop('acct')
    agg_in[numeric_cols_in] = agg_in[numeric_cols_in].fillna(0)


    # 4. 警示對手帳戶特徵
    print(" [6/9] 計算警示對手特徵...")
    agg_out_alert, agg_in_alert = calculate_alert_partner_features(df_txn, alert_set)

    # 5. 夜間交易特徵
    print(" [7/9] 計算夜間交易總量特徵...")
    df_night = df_txn[df_txn['time_period'] == 'night']
    agg_night_out = df_night.groupby("from_acct", observed=True).agg(night_out_count=("txn_amt", "count"), night_out_sum=("txn_amt", "sum")).rename_axis("acct").reset_index()
    agg_night_in = df_night.groupby("to_acct", observed=True).agg(night_in_count=("txn_amt", "count"), night_in_sum=("txn_amt", "sum")).rename_axis("acct").reset_index()

    # 6. 對手數量 / 自轉帳 / 類型比例
    print(" [8/9] 計算對手數量及複雜比例特徵 (Vectorized: 性能優化)...")
    agg_partners = df_txn.groupby("from_acct", observed=True)['to_acct'].nunique().reset_index(name="unique_out_partners").rename(columns={'from_acct': 'acct'})
    agg_partners_in = df_txn.groupby("to_acct", observed=True)['from_acct'].nunique().reset_index(name="unique_in_partners").rename(columns={'to_acct': 'acct'})
    
    # 建立Boolean籤
    # 在df_txn上新增臨時欄位，用於後續 groupby.agg
    df_txn['is_self_Y'] = (df_txn["is_self_txn"] == "Y").astype(int)
    df_txn['is_self_UNK'] = (df_txn["is_self_txn"] == "UNK").astype(int)
    df_txn['is_from_type_01'] = (df_txn["from_acct_type"] == "01").astype(int)
    df_txn['is_rare_partner'] = df_txn["to_acct"].isin(rare_accts).astype(int)
    df_txn['is_to_type_02'] = (df_txn["to_acct_type"] == "02").astype(int)

    # 透過單次 Groupby 提取多項平均與加總
    agg_self_type = df_txn.groupby("from_acct", observed=True).agg(
        self_txn_ratio=('is_self_Y', 'mean'),
        self_txn_unk_ratio=('is_self_UNK', 'mean'),
        out_acct_type_01_ratio=('is_from_type_01', 'mean'),
        Rare_Partner_Out_Count=('is_rare_partner', 'sum'),
        Out_to_Type_02_Count=('is_to_type_02', 'sum'),
    ).rename_axis("acct").reset_index()

    # 清理臨時欄位
    df_txn.drop(columns=['is_self_Y', 'is_self_UNK', 'is_from_type_01', 'is_rare_partner', 'is_to_type_02'], inplace=True)

    # 7. Crosstab 類別比例特徵：分析每個帳戶在特定交易管道或交易幣別的分佈組成
    print(" [9/9] 計算 Crosstab 類別比例特徵 (Progress Bar)...")
    crosstab_cols = ["time_period", "currency_type", "channel_type"]
    agg_crosstab_list = []
    for col in tqdm(crosstab_cols, desc=f" [9/9] Crosstab Features for {len(df_txn):,} txns"):
        agg_from = pd.crosstab(df_txn["from_acct"], df_txn[col], normalize="index").reset_index().rename(columns={'from_acct':'acct'})
        agg_to   = pd.crosstab(df_txn["to_acct"], df_txn[col], normalize="index").reset_index().rename(columns={'to_acct':'acct'})
        agg_crosstab_list.extend([agg_from, agg_to])

    # 8. 合併特徵 (修正重複合併問題 + 確保所有目標帳戶存在)
    df_features_result = pd.DataFrame({'acct': list(target_accts)}) 

    # 基礎特徵
    df_features_result = df_features_result.merge(agg_out, on="acct", how="left").merge(agg_in, on="acct", how="left")
    df_features_result = df_features_result.merge(agg_partners, on="acct", how="left").merge(agg_partners_in, on="acct", how="left")
    df_features_result = df_features_result.merge(agg_self_type, on="acct", how="left")
    
    # 風險特徵
    df_features_result = df_features_result.merge(agg_out_alert, on="acct", how="left").merge(agg_in_alert, on="acct", how="left")
    
    # 時間特徵
    df_features_result = df_features_result.merge(agg_night_out, on="acct", how="left").merge(agg_night_in, on="acct", how="left")
    
    # 優化特徵
    df_features_result = df_features_result.merge(agg_out_7d, on="acct", how="left")
    df_features_result = df_features_result.merge(agg_pacing, on="acct", how="left")
    df_features_result = df_features_result.merge(max_hourly_agg, on="acct", how="left") # 🚨 合併單小時最大交易筆數
    
    # Crosstab 特徵
    for agg_df in agg_crosstab_list:
        df_features_result = df_features_result.merge(agg_df, on="acct", how="left")
            
    # 修正 Type Error: 僅對數值型特徵進行 fillna(0)
    # 這一步將所有缺失的數值特徵（來自左合併）填補為 0
    numeric_cols_final = df_features_result.select_dtypes(include=[np.number]).columns
    df_features_result[numeric_cols_final] = df_features_result[numeric_cols_final].fillna(0)
    # df_features_result = df_features_result.fillna(0) # 原始危險的填補

    # 9. 衍生特徵計算 透過非線性關係交叉與平滑處理，放大高風險詐騙行為與常態行為的差距。
    total_sum = df_features_result['out_sum'] + df_features_result['in_sum']
    total_count = df_features_result['out_count'] + df_features_result['in_count']
    # 金流與交易次數淨額
    df_features_result['net_sum'] = df_features_result['out_sum'] - df_features_result['in_sum']
    df_features_result['net_count'] = df_features_result['out_count'] - df_features_result['in_count']
    df_features_result['sum_ratio'] = df_features_result['out_sum'] / (total_sum + 1e-6)
    df_features_result['count_ratio'] = df_features_result['out_count'] / (total_count + 1e-6)
    df_features_result['avg_amt_per_partner_out'] = df_features_result['out_sum'] / (df_features_result['unique_out_partners'] + 1e-6)
    df_features_result['avg_txn_per_partner_out'] = df_features_result['out_count'] / (df_features_result['unique_out_partners'] + 1e-6)
    df_features_result['has_txn'] = ((df_features_result['out_count'] + df_features_result['in_count']) > 0).astype(int)
    # 時間動態比值
    df_features_result['night_out_ratio'] = df_features_result['night_out_count'] / (df_features_result['out_count'] + 1e-6)
    df_features_result['night_in_ratio'] = df_features_result['night_in_count'] / (df_features_result['in_count'] + 1e-6)
    # 夜間交易金額佔總金額的比例 (高風險指標)
    df_features_result['night_out_amount_ratio'] = df_features_result['night_out_sum'] / (df_features_result['out_sum'] + 1e-6)
    # 速度與突發度指標
    df_features_result['out_count_7d_ratio'] = df_features_result['out_count_7d'] / (df_features_result['out_count'] + 1e-6)
    df_features_result['Rare_Partner_Out_Ratio'] = df_features_result['Rare_Partner_Out_Count'] / (df_features_result['out_count'] + 1e-6)
    df_features_result['Out_to_Type_02_Ratio'] = df_features_result['Out_to_Type_02_Count'] / (df_features_result['out_count'] + 1e-6)
    # 單小時最大筆數佔總筆數的比例 (集中度指標)
    df_features_result['Max_Hourly_Count_Ratio'] = df_features_result['Max_Hourly_Count'] / (df_features_result['out_count'] + 1e-6)
    
    return df_features_result

# -----------------------------------------------------
# 執行特徵計算 (分別計算訓練集和測試集特徵)
# -----------------------------------------------------
print("🚀 開始計算時序性特徵...")
# ⭐ 修正：用包含正負樣本的 train_candidate_accts 來生成特徵
df_features_train_set = create_features(df_transaction_train, alert_accts_set, train_candidate_accts)
df_features_test_set = create_features(df_transaction_test, alert_accts_set, test_accts)

print("✅ 特徵工程 (時序性計算) 完成。")

🚀 開始計算時序性特徵...
 [1/9] 計算 7 天內交易速度特徵...
 [2/9] 計算交易節奏 (間隔) 特徵 (Progress Bar)...


/tmp/ipykernel_3200/1223327036.py:26: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_txn_paced['txn_gap'] = df_txn_paced.groupby('from_acct')['txn_minute_of_day'].diff()


  0%|          | 0/813804 [00:00<?, ?it/s]

 [3/9] 計算單小時最大筆數特徵...


/tmp/ipykernel_3200/1223327036.py:38: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  hourly_max_count = df_txn.groupby(['from_acct', 'txn_hour']).size().reset_index(name='Hourly_Count')
/tmp/ipykernel_3200/1223327036.py:40: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  max_hourly_agg = hourly_max_count.groupby('from_acct')['Hourly_Count'].max().reset_index(name='Max_Hourly_Count').rename(columns={'from_acct': 'acct'})


...

  0%|          | 0/1169482 [00:00<?, ?it/s]

 [6/9] 計算警示對手特徵...
 [7/9] 計算夜間交易總量特徵...
 [8/9] 計算對手數量及複雜比例特徵 (Vectorized: 性能優化)...
 [9/9] 計算 Crosstab 類別比例特徵 (Progress Bar)...


 [9/9] Crosstab Features for 4,435,890 txns:   0%|          | 0/3 [00:00<?, ?it/s]

✅ 特徵工程 (時序性計算) 完成。


/tmp/ipykernel_3200/1223327036.py:169: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_features_result['night_out_amount_ratio'] = df_features_result['night_out_sum'] / (df_features_result['out_sum'] + 1e-6)
/tmp/ipykernel_3200/1223327036.py:171: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_features_result['out_count_7d_ratio'] = df_features_result['out_count_7d'] / (df_features_result['out_count'] + 1e-6)
/tmp/ipykernel_3200/1223327036.py:173: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result 

In [12]:
# =====================================================
# 9. 篩選目標帳戶 → 切分 train / test 
# =====================================================

# 1. 提取觀察期內活躍之行內候選母體
esun_accts_train = train_candidate_accts
print(f"✅ 訓練集候選帳戶數 (時間受限): {len(esun_accts_train):,}")

# ------------------- 訓練集 (Train Set) -------------------
# 依據活躍帳戶清單，過濾歷史特徵矩陣
train_features_final = df_features_train_set[df_features_train_set["acct"].isin(esun_accts_train)].copy()

# 標籤：透過常數時間雜湊比對，精準定義出正樣本（已觸發警示者=1）與未警示之常態負樣本（=0）
train_features_final["label"] = train_features_final["acct"].isin(df_alert["acct"]).astype(int) 
print(f"✅ (過濾 Esun/Active 後) 訓練集臨時尺寸: {train_features_final.shape}")


# 將所有測試集帳戶從訓練集中移除
overlap_accts = test_accts 
train_features = train_features_final[~train_features_final["acct"].isin(overlap_accts)].copy()
print(f"✅ 訓練集 (Features + Label) 尺寸: {train_features.shape} (已移除 {len(overlap_accts)} 筆重疊帳戶)")

# ------------------- 測試集 (Test Set) -------------------
# 測試集：使用完整交易資料計算出的特徵集
test_features_data = df_features_test_set[df_features_test_set["acct"].isin(test_accts)].copy()

# 確保測試集帳戶的順序與 df_predict 匹配
test_features = df_predict[["acct"]].merge(test_features_data, on="acct", how="left")

# 新增 pd.NA -> np.nan 轉換，確保 CatBoost 預測時不出錯
numeric_cols_test = test_features.select_dtypes(include=[np.number]).columns
test_features[numeric_cols_test] = test_features[numeric_cols_test].replace(pd.NA, np.nan) 
test_features[numeric_cols_test] = test_features[numeric_cols_test].fillna(0)

print(f"✅ 測試集 (Features only) 尺寸: {test_features.shape}")

# 訓練/測試重複帳戶數量
overlap = set(train_features["acct"]) & set(test_features["acct"])
print(f"✅ 訓練/測試重複帳戶數量: {len(overlap)}") 


✅ 訓練集候選帳戶數 (時間受限): 332,208
✅ (過濾 Esun/Active 後) 訓練集臨時尺寸: (332208, 106)
✅ 訓練集 (Features + Label) 尺寸: (327429, 106) (已移除 4780 筆重疊帳戶)
✅ 測試集 (Features only) 尺寸: (4780, 105)
✅ 訓練/測試重複帳戶數量: 0


In [13]:
#訓練集：train_features
#測試集：test_features
# 確認欄位：都一樣，Train多個標籤欄位
print("Train columns:", train_features.columns.tolist())
print("Test  columns:", test_features.columns.tolist())


Train columns: ['acct', 'out_sum', 'out_mean', 'out_std', 'out_max', 'out_count', 'out_cov', 'in_sum', 'in_mean', 'in_std', 'in_max', 'in_count', 'in_cov', 'unique_out_partners', 'unique_in_partners', 'self_txn_ratio', 'self_txn_unk_ratio', 'out_acct_type_01_ratio', 'Rare_Partner_Out_Count', 'Out_to_Type_02_Count', 'alert_partner_out_cnt', 'alert_partner_out_sum', 'alert_partner_in_cnt', 'alert_partner_in_sum', 'night_out_count', 'night_out_sum', 'night_in_count', 'night_in_sum', 'out_sum_7d', 'out_count_7d', 'txn_gap_min', 'txn_gap_std', 'Max_Hourly_Count', 'afternoon_x', 'evening_x', 'morning_x', 'night_x', 'afternoon_y', 'evening_y', 'morning_y', 'night_y', 'AUD_x', 'CAD_x', 'CHF_x', 'CNY_x', 'EUR_x', 'GBP_x', 'HKD_x', 'JPY_x', 'NZD_x', 'SEK_x', 'SGD_x', 'THB_x', 'TWD_x', 'USD_x', 'ZAR_x', 'MXN_x', 'AUD_y', 'CAD_y', 'CHF_y', 'CNY_y', 'EUR_y', 'GBP_y', 'HKD_y', 'JPY_y', 'NZD_y', 'SEK_y', 'SGD_y', 'THB_y', 'TWD_y', 'USD_y', 'ZAR_y', 'MXN_y', '01_x', '02_x', '03_x', '04_x', '05_x', '06

In [14]:
# 標籤分布
label_counts = train_features["label"].value_counts(normalize=False)
label_ratio = train_features["label"].value_counts(normalize=True)

print("📊 訓練集標籤分布")
print(label_counts)
print("\n📈 訓練集比例")
print(label_ratio)


📊 訓練集標籤分布
label
0    326425
1      1004
Name: count, dtype: int64

📈 訓練集比例
label
0    0.996934
1    0.003066
Name: proportion, dtype: float64


In [15]:
# ====== 存檔 ======
train_outfile = "/work/claire901114/初賽資料/train_features_1021.csv"
test_outfile  = "/work/claire901114/初賽資料/test_features_1021.csv"

train_features.to_csv(train_outfile, index=False, encoding="utf-8-sig")
test_features.to_csv(test_outfile, index=False, encoding="utf-8-sig")

print(f"✅ 訓練集已存檔: {train_outfile}, shape = {train_features.shape}")
print(f"✅ 測試集已存檔: {test_outfile}, shape = {test_features.shape}")


✅ 訓練集已存檔: /work/claire901114/初賽資料/train_features_1021.csv, shape = (327429, 106)
✅ 測試集已存檔: /work/claire901114/初賽資料/test_features_1021.csv, shape = (4780, 105)


# 讀分好的訓練與測試集

In [1]:
import pandas as pd
# ====== 檔案路徑 ======
train_outfile = "/work/claire901114/初賽資料/train_features_1021.csv"
test_outfile  = "/work/claire901114/初賽資料/test_features_1021.csv"

# ====== 讀取檔案 ======
train_features_final = pd.read_csv(train_outfile)
test_features_final  = pd.read_csv(test_outfile)

print("✅ 訓練集 shape:", train_features_final.shape)
print("✅ 測試集 shape:", test_features_final.shape)


✅ 訓練集 shape: (327429, 106)
✅ 測試集 shape: (4780, 105)


In [2]:
# 標籤分布
label_counts = train_features_final["label"].value_counts(normalize=False)
label_ratio = train_features_final["label"].value_counts(normalize=True)

print("📊 訓練集標籤分布")
print(label_counts)
print("\n📈 訓練集比例")
print(label_ratio)

📊 訓練集標籤分布
label
0    326425
1      1004
Name: count, dtype: int64

📈 訓練集比例
label
0    0.996934
1    0.003066
Name: proportion, dtype: float64


# CatBoost （完整特徵＋RandomUnderSampler）

In [19]:
import pandas as pd
import numpy as np
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import StratifiedShuffleSplit
from imblearn.under_sampling import EditedNearestNeighbours 
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier
from tqdm import tqdm

# =====================================================
# 1. 拆分資料 (保持原樣，僅定義 X/y)
# =====================================================
X = train_features_final.drop(columns=["acct","label"]) # 假設已使用 train_features_final
y = train_features_final["label"]
# 確認維度
print(f"✅ X (特徵集) 尺寸: {X.shape}")
print(f"✅ y (標籤集) 尺寸: {y.shape}")
print(f"✅ X 欄位數 (特徵數): {X.shape[1]}")

# =====================================================
# 2. 定義超參數搜尋空間
# =====================================================
def sample_params(seed):
    rng = np.random.RandomState(seed)
    return {
        "depth": rng.randint(4, 10),                     # 相當於 max_depth
        "learning_rate": rng.uniform(0.01, 0.3),
        "l2_leaf_reg": rng.uniform(1, 10),               # L2 正則化
        "bagging_temperature": rng.uniform(0, 1),        # 隨機 bagging 控制
        "iterations": 200,                               # 固定 
        "random_seed": seed
    }

# =====================================================
# 3. 重複 Stratified 抽樣 + Undersampling 訓練
# =====================================================
n_search_iterations = 500 # 迭代次數
search_results = []
best_params_list = []
best_f1_score = -1
final_params = {}

for i in tqdm(range(n_search_iterations), desc="Repeated Stratified Search"):
    
    # 每次迭代執行一次分層抽樣 (70% Train / 30% Validation)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42 + i)
    train_idx, val_idx = next(sss.split(X, y))

    X_train_split, y_train_split = X.iloc[train_idx], y.iloc[train_idx]
    X_val_split, y_val_split = X.iloc[val_idx], y.iloc[val_idx]
    
    # === 欠採樣 ===
    # 用 RandomUnderSampler 將正負樣本比例控制在 1:10
    rus = RandomUnderSampler(sampling_strategy=0.1, random_state=42+i) 
    
    # 僅對「當前 Fold 的訓練集」進行重採樣
    X_train_res, y_train_res = rus.fit_resample(X_train_split, y_train_split)
    

    
    # === 抽一組超參數 ===
    params = sample_params(seed=42+i)

    model = CatBoostClassifier(
        **params,
        eval_metric="F1",
        verbose=0,
        task_type="CPU",
        thread_count=-1  
    )

    # 在欠採樣後的訓練集上訓練
    model.fit(X_train_res, y_train_res)

    # 在驗證集評估 F1 Score
    y_pred = model.predict(X_val_split)
    f1 = f1_score(y_val_split, y_pred)

    search_results.append(f1)
    best_params_list.append(params)
    
    if f1 > best_f1_score:
        best_f1_score = f1
        final_params = params


# =====================================================
# 4. 統計最佳參數
# =====================================================
mean_f1 = np.mean(search_results)
ci_lower = np.percentile(search_results, 2.5)
ci_upper = np.percentile(search_results, 97.5)

print(f"\n✅ {n_search_iterations} 次重複分層搜尋 F1 平均值: {mean_f1:.4f}")
print(f"✅ 95% 信賴區間: [{ci_lower:.4f}, {ci_upper:.4f}]")

print("✅ 最佳 F1:", best_f1_score)
print("✅ 最佳超參數組合:", final_params)

✅ X (特徵集) 尺寸: (327429, 104)
✅ y (標籤集) 尺寸: (327429,)
✅ X 欄位數 (特徵數): 104


Repeated Stratified Search: 100%|██████████| 500/500 [18:23<00:00,  2.21s/it]


✅ 500 次重複分層搜尋 F1 平均值: 0.2322
✅ 95% 信賴區間: [0.2075, 0.2576]
✅ 最佳 F1: 0.2832080200501253
✅ 最佳超參數組合: {'depth': 6, 'learning_rate': 0.02758349376244848, 'l2_leaf_reg': 1.613433624312349, 'bagging_temperature': 0.23771355882147815, 'iterations': 200, 'random_seed': 260}


In [20]:
import pandas as pd
import numpy as np
from imblearn.under_sampling import RandomUnderSampler
from catboost import CatBoostClassifier

# =====================================================
# 5. 最終模型訓練 (全量資料 - 修正為直接使用 RUS)
# =====================================================

print("🚀 開始對全量訓練資料進行欠採樣 (直接使用 RUS 調整比例)...")

# 將負樣本比例調整為正樣本的 10 倍 (0.1)
rus = RandomUnderSampler(
    sampling_strategy=0.1, 
    random_state=42
) 
X_final, y_final = rus.fit_resample(X, y) # 直接對原始 X, y 採樣

print(f"✅ 欠採樣完成。最終訓練樣本數: {len(X_final):,} (正樣本數: {y_final.sum():,})")


# 最終 CatBoost 模型訓練
# 確保 final_params 包含 thread_count=-1
final_model = CatBoostClassifier(
    **final_params,
    eval_metric="F1",
    task_type="CPU",
    verbose=0,
    thread_count=-1,       
    allow_writing_files=False 
)

print("\n🚀 開始訓練最終 CatBoost 模型 (使用最佳超參數)...")
final_model.fit(X_final, y_final)

# 儲存模型
model_filename = "final_model_CatBoost_1021.cbm"
final_model.save_model(model_filename)


print("🎉 最終 CatBoost 模型已完成訓練")
print(f"✅ 模型已儲存為檔案：{model_filename}")

🚀 開始對全量訓練資料進行欠採樣 (直接使用 RUS 調整比例)...
✅ 欠採樣完成。最終訓練樣本數: 11,044 (正樣本數: 1,004)

🚀 開始訓練最終 CatBoost 模型 (使用最佳超參數)...
🎉 最終 CatBoost 模型已完成訓練
✅ 模型已儲存為檔案：final_model_CatBoost_1021.cbm


In [21]:
# =====================================================
# 6. 產生測試集預測與 submission
# =====================================================
X_test = test_features.drop(columns=["acct"])
y_test_pred = final_model.predict(X_test)

submission = pd.DataFrame({
    "acct": test_features["acct"],
    "label": y_test_pred
})

print("🎉 已完成預測，submission DataFrame 生成成功")
print(submission.head())



🎉 已完成預測，submission DataFrame 生成成功
                                                acct  label
0  fcf31c5113d3dbd9cb5056045c6a0f213bd8a4fc1bc834...      0
1  e21dfa45e990364194468e501fbfe52ec02a4b71a2e2e8...      0
2  2552e943aaf9caa33183758cd40128ef20a6e6ff16c232...      0
3  71700e7b7c3d40abdfdbcc7afc0752fa8d9bd28b408651...      0
4  c70349fc718ffb88f03f31b5a7fcf65b33dd71dce6fee0...      0


In [22]:
# 確認與範例對齊並存檔¶
submission

,acct,label
0,fcf31c5113d3dbd9cb5056045c6a0f213bd8a4fc1bc834...,0
1,e21dfa45e990364194468e501fbfe52ec02a4b71a2e2e8...,0
2,2552e943aaf9caa33183758cd40128ef20a6e6ff16c232...,0
3,71700e7b7c3d40abdfdbcc7afc0752fa8d9bd28b408651...,0
4,c70349fc718ffb88f03f31b5a7fcf65b33dd71dce6fee0...,0
...,...,...
4775,09747f71cc6234a75312f2e77f79b9a51e5145e114e8b0...,0
4776,32e6bf3ca071af026794f4df028f3cdaa43ddd499bac10...,0
4777,e9a6861c68821da506be76419df559efc167ef8a056063...,0
4778,75ca5a3798f3cad4c0e1c1fd0adca48f020b2bda856a8e...,0


In [23]:
file_sub = "submission_template.csv"
submission_tem = pd.read_csv(file_sub)
submission_tem

,acct,label
0,fcf31c5113d3dbd9cb5056045c6a0f213bd8a4fc1bc834...,0
1,e21dfa45e990364194468e501fbfe52ec02a4b71a2e2e8...,0
2,2552e943aaf9caa33183758cd40128ef20a6e6ff16c232...,0
3,71700e7b7c3d40abdfdbcc7afc0752fa8d9bd28b408651...,0
4,c70349fc718ffb88f03f31b5a7fcf65b33dd71dce6fee0...,0
...,...,...
4775,09747f71cc6234a75312f2e77f79b9a51e5145e114e8b0...,0
4776,32e6bf3ca071af026794f4df028f3cdaa43ddd499bac10...,0
4777,e9a6861c68821da506be76419df559efc167ef8a056063...,0
4778,75ca5a3798f3cad4c0e1c1fd0adca48f020b2bda856a8e...,0


In [24]:
# 1. 確認數量是否一樣
print("df_predict:", submission_tem["acct"].nunique())
print("submission:", submission["acct"].nunique())

# 2. 確認 acct 集合是否一樣
same_set = set(submission_tem["acct"]) == set(submission["acct"])
print("兩個集合是否完全相同:", same_set)

# 3. 確認順序是否一樣
same_order = submission_tem["acct"].tolist() == submission["acct"].tolist()
print("兩個 acct 是否順序完全相同:", same_order)


df_predict: 4780
submission: 4780
兩個集合是否完全相同: True
兩個 acct 是否順序完全相同: True


In [25]:
# 重新排序 submission
submission = submission.set_index("acct").loc[submission_tem["acct"]].reset_index()

# 再確認一次
print(submission.head())
print(submission_tem.head())


                                                acct  label
0  fcf31c5113d3dbd9cb5056045c6a0f213bd8a4fc1bc834...      0
1  e21dfa45e990364194468e501fbfe52ec02a4b71a2e2e8...      0
2  2552e943aaf9caa33183758cd40128ef20a6e6ff16c232...      0
3  71700e7b7c3d40abdfdbcc7afc0752fa8d9bd28b408651...      0
4  c70349fc718ffb88f03f31b5a7fcf65b33dd71dce6fee0...      0
                                                acct  label
0  fcf31c5113d3dbd9cb5056045c6a0f213bd8a4fc1bc834...      0
1  e21dfa45e990364194468e501fbfe52ec02a4b71a2e2e8...      0
2  2552e943aaf9caa33183758cd40128ef20a6e6ff16c232...      0
3  71700e7b7c3d40abdfdbcc7afc0752fa8d9bd28b408651...      0
4  c70349fc718ffb88f03f31b5a7fcf65b33dd71dce6fee0...      0


In [26]:
num_ones = (submission["label"] == 1).sum()
num_zeros = (submission["label"] == 0).sum()

print("label=1 筆數:", num_ones)
print("label=0 筆數:", num_zeros)
print("總筆數:", len(submission))



label=1 筆數: 277
label=0 筆數: 4503
總筆數: 4780


In [27]:
# 結果存檔
submission.to_csv("submission_CatBoost_1021.csv", index=False)

## CatBoost 特徵篩選

In [28]:
# 特徵篩選
from catboost import CatBoostClassifier
import pandas as pd

# 模型路徑
model_path = "final_model_CatBoost_1021.cbm"

# 載入模型
loaded_model = CatBoostClassifier()
loaded_model.load_model(model_path)

print("✅ CatBoost 模型載入成功！")

✅ CatBoost 模型載入成功！


In [29]:
import numpy as np

# 確保順序與訓練時一致
feature_names = list(train_features.drop(columns=["acct", "label"]).columns)

# 取得特徵重要性（Feature Importance）
importances = loaded_model.get_feature_importance(type="FeatureImportance")

# 組成 DataFrame
feat_imp_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

display(feat_imp_df.head(50))


,Feature,Importance
13,unique_in_partners,20.276606
89,UNK_y,13.940060
27,out_sum_7d,5.834480
9,in_max,4.847284
101,Rare_Partner_Out_Ratio,3.834262
100,out_count_7d_ratio,3.361658
102,Out_to_Type_02_Ratio,3.255727
38,morning_y,3.250540
30,txn_gap_std,2.938688
90,net_sum,2.906753


# CatBoost 特徵篩選後（用前22重要特徵＋RandomUnderSampler）

In [30]:
top22_features = [
    "unique_in_partners", "UNK_y", "out_sum_7d", "in_max", "Rare_Partner_Out_Ratio",
    "out_count_7d_ratio", "Out_to_Type_02_Ratio", "morning_y", "txn_gap_std", "net_sum",
    "Rare_Partner_Out_Count", "txn_gap_min", "in_sum", "in_mean", "04_y",
    "out_count_7d", "self_txn_unk_ratio", "in_count", "out_max", "UNK_x",
    "sum_ratio", "afternoon_y"
]


In [35]:
import pandas as pd
import numpy as np
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import StratifiedShuffleSplit
from imblearn.under_sampling import EditedNearestNeighbours
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier
from tqdm import tqdm

# =====================================================
# 1. 拆分資料 (保持原樣，僅定義 X/y)
# =====================================================
# 只保留前 22 特徵作為輸入
X = train_features_final[top22_features].copy()
y = train_features_final["label"].copy()

# 確認維度
print(f"✅ X (特徵集) 尺寸: {X.shape}")
print(f"✅ y (標籤集) 尺寸: {y.shape}")
print(f"✅ X 欄位數 (特徵數): {X.shape[1]}")

# =====================================================
# 2. 超參數搜尋空間
# =====================================================
def sample_params(seed):
    rng = np.random.RandomState(seed)
    return {
        "depth": rng.randint(4, 10),                     # 相當於 max_depth
        "learning_rate": rng.uniform(0.01, 0.3),
        "l2_leaf_reg": rng.uniform(1, 10),               # L2 正則化
        "bagging_temperature": rng.uniform(0, 1),        # 隨機 bagging 控制
        "iterations": 200,                             
        "random_seed": seed
    }

# =====================================================
# 3. 重複 Stratified 抽樣 + Undersampling 訓練
# =====================================================
n_search_iterations = 500 # 保持原來的 500 次迭代
search_results = []
best_params_list = []
best_f1_score = -1
final_params = {}

for i in tqdm(range(n_search_iterations), desc="Repeated Stratified Search"):
    
    # 每次迭代執行一次分層抽樣 (70% Train / 30% Validation)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42 + i)
    train_idx, val_idx = next(sss.split(X, y))

    X_train_split, y_train_split = X.iloc[train_idx], y.iloc[train_idx]
    X_val_split, y_val_split = X.iloc[val_idx], y.iloc[val_idx]

    # 使用 RandomUnderSampler 對原始訓練集進行比例調整 (1:10)
    rus = RandomUnderSampler(sampling_strategy=0.1, random_state=42+i) 
    
    # 將 RUS 應用在分層抽樣後的訓練集 X_train_split, y_train_split 上
    X_train_res, y_train_res = rus.fit_resample(X_train_split, y_train_split)
    

    
    # === 抽一組超參數 ===
    params = sample_params(seed=42+i)

    model = CatBoostClassifier(
        **params,
        eval_metric="F1",
        verbose=0,
        task_type="CPU",
        thread_count=-1 
    )

    # 在欠採樣後的訓練集上訓練
    model.fit(X_train_res, y_train_res)

    # 在驗證集 評估 F1 Score
    y_pred = model.predict(X_val_split)
    f1 = f1_score(y_val_split, y_pred)

    search_results.append(f1)
    best_params_list.append(params)
    
    if f1 > best_f1_score:
        best_f1_score = f1
        final_params = params


# =====================================================
# 4. 統計最佳參數
# =====================================================
mean_f1 = np.mean(search_results)
ci_lower = np.percentile(search_results, 2.5)
ci_upper = np.percentile(search_results, 97.5)

print(f"\n✅ {n_search_iterations} 次重複分層搜尋 F1 平均值: {mean_f1:.4f}")
print(f"✅ 95% 信賴區間: [{ci_lower:.4f}, {ci_upper:.4f}]")

print("✅ 最佳 F1:", best_f1_score)
print("✅ 最佳超參數組合:", final_params)

✅ X (特徵集) 尺寸: (327429, 22)
✅ y (標籤集) 尺寸: (327429,)
✅ X 欄位數 (特徵數): 22


Repeated Stratified Search: 100%|██████████| 500/500 [09:23<00:00,  1.13s/it]


✅ 500 次重複分層搜尋 F1 平均值: 0.2196
✅ 95% 信賴區間: [0.1981, 0.2423]
✅ 最佳 F1: 0.2600116076610563
✅ 最佳超參數組合: {'depth': 6, 'learning_rate': 0.02758349376244848, 'l2_leaf_reg': 1.613433624312349, 'bagging_temperature': 0.23771355882147815, 'iterations': 200, 'random_seed': 260}


In [37]:
import pandas as pd
import numpy as np
from imblearn.under_sampling import RandomUnderSampler
from catboost import CatBoostClassifier

# =====================================================
# 5. 最終模型訓練
# =====================================================

print("🚀 開始對全量訓練資料進行欠採樣 (直接使用 RUS 調整比例)...")

# 將負樣本比例調整為正樣本的 10 倍 (0.1)
rus = RandomUnderSampler(
    sampling_strategy=0.1, 
    random_state=42
) 
X_final, y_final = rus.fit_resample(X, y) # 直接對原始 X, y 採樣

print(f"✅ 欠採樣完成。最終訓練樣本數: {len(X_final):,} (正樣本數: {y_final.sum():,})")


# 2. 最終 CatBoost 模型訓練
# 確保 final_params 包含 thread_count=-1
final_model = CatBoostClassifier(
    **final_params,
    eval_metric="F1",
    task_type="CPU",
    verbose=0,
    thread_count=-1,       
    allow_writing_files=False 
)

print("\n🚀 開始訓練最終 CatBoost 模型 (使用最佳超參數)...")
final_model.fit(X_final, y_final)

# 儲存模型
model_filename = "final_model_CatBoost_1021_top30.cbm"
final_model.save_model(model_filename)


print("🎉 最終 CatBoost 模型已完成訓練")
print(f"✅ 模型已儲存為檔案：{model_filename}")

🚀 開始對全量訓練資料進行欠採樣 (直接使用 RUS 調整比例)...
✅ 欠採樣完成。最終訓練樣本數: 11,044 (正樣本數: 1,004)

🚀 開始訓練最終 CatBoost 模型 (使用最佳超參數)...
🎉 最終 CatBoost 模型已完成訓練
✅ 模型已儲存為檔案：final_model_CatBoost_1021_top30.cbm


In [38]:
# =====================================================
# 6. 產生測試集預測與 submission
# =====================================================
X_test = test_features.drop(columns=["acct"])
y_test_pred = final_model.predict(X_test)

submission = pd.DataFrame({
    "acct": test_features["acct"],
    "label": y_test_pred
})

print("🎉 已完成預測，submission DataFrame 生成成功")
print(submission.head())


🎉 已完成預測，submission DataFrame 生成成功
                                                acct  label
0  fcf31c5113d3dbd9cb5056045c6a0f213bd8a4fc1bc834...      0
1  e21dfa45e990364194468e501fbfe52ec02a4b71a2e2e8...      0
2  2552e943aaf9caa33183758cd40128ef20a6e6ff16c232...      0
3  71700e7b7c3d40abdfdbcc7afc0752fa8d9bd28b408651...      0
4  c70349fc718ffb88f03f31b5a7fcf65b33dd71dce6fee0...      0


In [39]:
num_ones = (submission["label"] == 1).sum()
num_zeros = (submission["label"] == 0).sum()

print("label=1 筆數:", num_ones)
print("label=0 筆數:", num_zeros)
print("總筆數:", len(submission))


label=1 筆數: 296
label=0 筆數: 4484
總筆數: 4780


In [40]:
file_sub = "submission_template.csv"
submission_tem = pd.read_csv(file_sub)
submission_tem

,acct,label
0,fcf31c5113d3dbd9cb5056045c6a0f213bd8a4fc1bc834...,0
1,e21dfa45e990364194468e501fbfe52ec02a4b71a2e2e8...,0
2,2552e943aaf9caa33183758cd40128ef20a6e6ff16c232...,0
3,71700e7b7c3d40abdfdbcc7afc0752fa8d9bd28b408651...,0
4,c70349fc718ffb88f03f31b5a7fcf65b33dd71dce6fee0...,0
...,...,...
4775,09747f71cc6234a75312f2e77f79b9a51e5145e114e8b0...,0
4776,32e6bf3ca071af026794f4df028f3cdaa43ddd499bac10...,0
4777,e9a6861c68821da506be76419df559efc167ef8a056063...,0
4778,75ca5a3798f3cad4c0e1c1fd0adca48f020b2bda856a8e...,0


In [41]:
# 1. 確認數量是否一樣
print("df_predict:", submission_tem["acct"].nunique())
print("submission:", submission["acct"].nunique())

# 2. 確認 acct 集合是否一樣
same_set = set(submission_tem["acct"]) == set(submission["acct"])
print("兩個集合是否完全相同:", same_set)

# 3. 確認順序是否一樣
same_order = submission_tem["acct"].tolist() == submission["acct"].tolist()
print("兩個 acct 是否順序完全相同:", same_order)

df_predict: 4780
submission: 4780
兩個集合是否完全相同: True
兩個 acct 是否順序完全相同: True


In [42]:
submission.to_csv("submission_CatBoost_1021_top30.csv", index=False)

# CatBoost （完整特徵＋SMOTE與RandomUnderSampler）

In [43]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier
from tqdm import tqdm

# =====================================================
# 1. 拆分資料
# =====================================================
X = train_features_final.drop(columns=["acct","label"]) # 假設已使用 train_features_final
y = train_features_final["label"]
# 確認維度
print(f"✅ X (特徵集) 尺寸: {X.shape}")
print(f"✅ y (標籤集) 尺寸: {y.shape}")
print(f"✅ X 欄位數 (特徵數): {X.shape[1]}")

# =====================================================
# 2. 超參數搜尋空間
# =====================================================
def sample_params(seed):
    rng = np.random.RandomState(seed)
    return {
        "depth": rng.randint(4, 10),                     # 相當於 max_depth
        "learning_rate": rng.uniform(0.01, 0.3),
        "l2_leaf_reg": rng.uniform(1, 10),               # L2 正則化
        "bagging_temperature": rng.uniform(0, 1),        # 隨機 bagging 控制
        "iterations": 200,                               
        "random_seed": seed
    }

# =====================================================
# 3. 🚨 修正：重複 Stratified 抽樣 + Undersampling 訓練
# =====================================================
n_search_iterations = 500 # 500 次迭代
search_results = []
best_params_list = []
best_f1_score = -1
final_params = {}

for i in tqdm(range(n_search_iterations), desc="Repeated Stratified Search"):
    
    # 每次迭代執行一次分層抽樣 (70% Train / 30% Validation)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42 + i)
    train_idx, val_idx = next(sss.split(X, y))

    X_train_split, y_train_split = X.iloc[train_idx], y.iloc[train_idx]
    X_val_split, y_val_split = X.iloc[val_idx], y.iloc[val_idx]
    
    # 步驟 1: 使用 SMOTE 將少數類別的樣本數增加到多數類別的 10%
    # sampling_strategy=0.1 表示 n_minority / n_majority = 0.1
    over = SMOTE(sampling_strategy=0.1, random_state=42+i)
    
    # 步驟 2: 使用 RandomUnderSampler 將多數類別減少，使最終比例約為 1:2
    # sampling_strategy=0.5 表示 n_minority / n_majority = 0.5
    under = RandomUnderSampler(sampling_strategy=0.5, random_state=42+i)

    X_temp, y_temp = over.fit_resample(X_train_split, y_train_split)
    X_train_res, y_train_res = under.fit_resample(X_temp, y_temp)

    
    # === 抽一組超參數 ===
    params = sample_params(seed=42+i)

    model = CatBoostClassifier(
        **params,
        eval_metric="F1",
        verbose=0,
        task_type="CPU",
        thread_count=-1  
    )

    # 在欠採樣後的訓練集上訓練
    model.fit(X_train_res, y_train_res)

    # 在30% 驗證集上評估 F1 Score
    y_pred = model.predict(X_val_split)
    f1 = f1_score(y_val_split, y_pred)

    search_results.append(f1)
    best_params_list.append(params)
    
    if f1 > best_f1_score:
        best_f1_score = f1
        final_params = params


# =====================================================
# 4. 統計最佳參數
# =====================================================
mean_f1 = np.mean(search_results)
ci_lower = np.percentile(search_results, 2.5)
ci_upper = np.percentile(search_results, 97.5)

print(f"\n✅ {n_search_iterations} 次重複分層搜尋 F1 平均值: {mean_f1:.4f}")
print(f"✅ 95% 信賴區間: [{ci_lower:.4f}, {ci_upper:.4f}]")

print("✅ 最佳 F1:", best_f1_score)
print("✅ 最佳超參數組合:", final_params)

✅ X (特徵集) 尺寸: (327429, 104)
✅ y (標籤集) 尺寸: (327429,)
✅ X 欄位數 (特徵數): 104


Repeated Stratified Search: 100%|██████████| 500/500 [41:45<00:00,  5.01s/it] 


✅ 500 次重複分層搜尋 F1 平均值: 0.3001
✅ 95% 信賴區間: [0.1404, 0.3739]
✅ 最佳 F1: 0.40670391061452515
✅ 最佳超參數組合: {'depth': 7, 'learning_rate': 0.2301538621498312, 'l2_leaf_reg': 3.0723718239356907, 'bagging_temperature': 0.564190131926705, 'iterations': 200, 'random_seed': 444}


In [44]:
import pandas as pd
import numpy as np
from imblearn.under_sampling import RandomUnderSampler
from catboost import CatBoostClassifier
# =====================================================
# 5. 最終模型訓練 
# =====================================================

print("🚀 開始對全量訓練資料進行混合採樣 (SMOTE + RUS)...")

# 步驟一 (過採樣): 將正樣本數增加到負樣本數的 10%
over = SMOTE(sampling_strategy=0.1, random_state=42)

# 步驟二 (欠採樣): 將負樣本數減少到正樣本數的 2 倍 (最終比例 1:2)
under = RandomUnderSampler(sampling_strategy=0.5, random_state=42)

X_temp, y_temp = over.fit_resample(X, y)
X_final, y_final = under.fit_resample(X_temp, y_temp)

print(f"✅ 混合採樣完成。最終訓練樣本數: {len(X_final):,}")
print("最終標籤分佈:\n", y_final.value_counts())
# 最終模型訓練
final_model = CatBoostClassifier(
    **final_params,
    eval_metric="F1",
    task_type="CPU",
    verbose=0,
    thread_count=-1,       # 設定使用所有 CPU
    allow_writing_files=False 
)

print("\n🚀 開始訓練最終 CatBoost 模型 (使用最佳超參數)...")
final_model.fit(X_final, y_final)

# 存模型
model_filename = "final_model_CatBoost_1021_SMOTEandRandomUnderSampler.cbm"
final_model.save_model(model_filename)


print("🎉 最終 CatBoost 模型已完成訓練")
print(f"✅ 模型已儲存為檔案：{model_filename}")

🚀 開始對全量訓練資料進行混合採樣 (SMOTE + RUS)...
✅ 混合採樣完成。最終訓練樣本數: 97,926
最終標籤分佈:
 label
0    65284
1    32642
Name: count, dtype: int64

🚀 開始訓練最終 CatBoost 模型 (使用最佳超參數)...
🎉 最終 CatBoost 模型已完成訓練
✅ 模型已儲存為檔案：final_model_CatBoost_1021_SMOTEandRandomUnderSampler.cbm


In [45]:
# =====================================================
# 6. 產生測試集預測與 submission
# =====================================================
X_test = test_features_final.drop(columns=["acct"])
y_test_pred = final_model.predict(X_test)

submission = pd.DataFrame({
    "acct": test_features_final["acct"],
    "label": y_test_pred
})

print("🎉 已完成預測，submission DataFrame 生成成功")
print(submission.head())


🎉 已完成預測，submission DataFrame 生成成功
                                                acct  label
0  fcf31c5113d3dbd9cb5056045c6a0f213bd8a4fc1bc834...      0
1  e21dfa45e990364194468e501fbfe52ec02a4b71a2e2e8...      0
2  2552e943aaf9caa33183758cd40128ef20a6e6ff16c232...      0
3  71700e7b7c3d40abdfdbcc7afc0752fa8d9bd28b408651...      0
4  c70349fc718ffb88f03f31b5a7fcf65b33dd71dce6fee0...      0


In [46]:
num_ones = (submission["label"] == 1).sum()
num_zeros = (submission["label"] == 0).sum()

print("label=1 筆數:", num_ones)
print("label=0 筆數:", num_zeros)
print("總筆數:", len(submission))

label=1 筆數: 108
label=0 筆數: 4672
總筆數: 4780


In [47]:
file_sub = "submission_template.csv"
submission_tem = pd.read_csv(file_sub)
submission_tem

,acct,label
0,fcf31c5113d3dbd9cb5056045c6a0f213bd8a4fc1bc834...,0
1,e21dfa45e990364194468e501fbfe52ec02a4b71a2e2e8...,0
2,2552e943aaf9caa33183758cd40128ef20a6e6ff16c232...,0
3,71700e7b7c3d40abdfdbcc7afc0752fa8d9bd28b408651...,0
4,c70349fc718ffb88f03f31b5a7fcf65b33dd71dce6fee0...,0
...,...,...
4775,09747f71cc6234a75312f2e77f79b9a51e5145e114e8b0...,0
4776,32e6bf3ca071af026794f4df028f3cdaa43ddd499bac10...,0
4777,e9a6861c68821da506be76419df559efc167ef8a056063...,0
4778,75ca5a3798f3cad4c0e1c1fd0adca48f020b2bda856a8e...,0


In [48]:
# 1. 確認數量是否一樣
print("df_predict:", submission_tem["acct"].nunique())
print("submission:", submission["acct"].nunique())

# 2. 確認 acct 集合是否一樣
same_set = set(submission_tem["acct"]) == set(submission["acct"])
print("兩個集合是否完全相同:", same_set)

# 3. 確認順序是否一樣
same_order = submission_tem["acct"].tolist() == submission["acct"].tolist()
print("兩個 acct 是否順序完全相同:", same_order)

df_predict: 4780
submission: 4780
兩個集合是否完全相同: True
兩個 acct 是否順序完全相同: True


In [49]:
submission.to_csv("submission_CatBoost_1021__SMOTEandRandomUnderSampler.csv", index=False)

In [50]:
# 特徵重要性分析
# 特徵篩選
from catboost import CatBoostClassifier
import pandas as pd

# 模型路徑
model_path = "final_model_CatBoost_1021_SMOTEandRandomUnderSampler.cbm"

# 載入模型
loaded_model = CatBoostClassifier()
loaded_model.load_model(model_path)

print("✅ CatBoost 模型載入成功！")

✅ CatBoost 模型載入成功！


In [51]:
import numpy as np

# 確保順序與訓練時一致
feature_names = list(train_features.drop(columns=["acct", "label"]).columns)

# 取得特徵重要性（Feature Importance）
importances = loaded_model.get_feature_importance(type="FeatureImportance")

# 組成 DataFrame
feat_imp_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

display(feat_imp_df.head(50))

,Feature,Importance
13,unique_in_partners,10.876885
31,Max_Hourly_Count,8.738724
25,night_in_count,7.833719
89,UNK_y,7.283228
29,txn_gap_min,5.072045
28,out_count_7d,4.024193
15,self_txn_unk_ratio,3.538797
23,night_out_count,3.516221
9,in_max,3.496116
90,net_sum,2.908755


# CatBoost （用前20重要特徵＋SMOTE與RandomUnderSampler）

In [56]:
top20_features = [
    'unique_in_partners',
    'Max_Hourly_Count',
    'night_in_count',
    'UNK_y',
    'txn_gap_min',
    'out_count_7d',
    'self_txn_unk_ratio',
    'night_out_count',
    'in_max',
    'net_sum',
    'in_count',
    'Rare_Partner_Out_Count',
    '03_y',
    'txn_gap_std',
    'UNK_x',
    'out_count',
    'Rare_Partner_Out_Ratio',
    'night_out_ratio',
    'in_mean',
    'night_in_sum'  
]

In [57]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier
from tqdm import tqdm

# =====================================================
# 1. 拆分資料 (保持原樣，僅定義 X/y)
# =====================================================
# 只保留前 20 特徵作為輸入
X = train_features_final[top20_features].copy()
y = train_features_final["label"].copy()

# 確認維度
print(f"✅ X (特徵集) 尺寸: {X.shape}")
print(f"✅ y (標籤集) 尺寸: {y.shape}")
print(f"✅ X 欄位數 (特徵數): {X.shape[1]}")

# =====================================================
# 2. 超參數搜尋空間 (保持原樣)
# =====================================================
def sample_params(seed):
    rng = np.random.RandomState(seed)
    return {
        "depth": rng.randint(4, 10),               
        "learning_rate": rng.uniform(0.01, 0.3),
        "l2_leaf_reg": rng.uniform(1, 10),               
        "bagging_temperature": rng.uniform(0, 1),        
        "iterations": 200,                               
        "random_seed": seed
    }

# =====================================================
# 3. 重複 Stratified 抽樣 + SMOTE/Undersampling 訓練
# =====================================================
n_search_iterations = 500 # 500 次迭代
search_results = []
best_params_list = []
best_f1_score = -1
final_params = {}

for i in tqdm(range(n_search_iterations), desc="Repeated Stratified Search"):
    
    # 每次迭代執行一次分層抽樣 (70% Train / 30% Validation)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42 + i)
    train_idx, val_idx = next(sss.split(X, y))

    X_train_split, y_train_split = X.iloc[train_idx], y.iloc[train_idx]
    X_val_split, y_val_split = X.iloc[val_idx], y.iloc[val_idx]
    
    # 步驟 1: 使用 SMOTE 將少數類別的樣本數增加到多數類別的 10%
    # sampling_strategy=0.1 表示 n_minority / n_majority = 0.1
    over = SMOTE(sampling_strategy=0.2, random_state=42+i)
    
    # 步驟 2: 使用 RandomUnderSampler 將多數類別減少，使最終比例約為 1:2
    # sampling_strategy=0.5 表示 n_minority / n_majority = 0.5
    under = RandomUnderSampler(sampling_strategy=1.0, random_state=42+i)

    X_temp, y_temp = over.fit_resample(X_train_split, y_train_split)
    X_train_res, y_train_res = under.fit_resample(X_temp, y_temp)

    
    # === 抽一組超參數 ===
    params = sample_params(seed=42+i)

    model = CatBoostClassifier(
        **params,
        eval_metric="F1",
        verbose=0,
        task_type="CPU",
        thread_count=-1  
    )

    # 在欠採樣後的訓練集上訓練
    model.fit(X_train_res, y_train_res)

    # 在驗證集上評估 F1 Score
    y_pred = model.predict(X_val_split)
    f1 = f1_score(y_val_split, y_pred)

    search_results.append(f1)
    best_params_list.append(params)
    
    if f1 > best_f1_score:
        best_f1_score = f1
        final_params = params


# =====================================================
# 4. 統計最佳參數
# =====================================================
mean_f1 = np.mean(search_results)
ci_lower = np.percentile(search_results, 2.5)
ci_upper = np.percentile(search_results, 97.5)

print(f"\n✅ {n_search_iterations} 次重複分層搜尋 F1 平均值: {mean_f1:.4f}")
print(f"✅ 95% 信賴區間: [{ci_lower:.4f}, {ci_upper:.4f}]")

print("✅ 最佳 F1:", best_f1_score)
print("✅ 最佳超參數組合:", final_params)

✅ X (特徵集) 尺寸: (327429, 20)
✅ y (標籤集) 尺寸: (327429,)
✅ X 欄位數 (特徵數): 20


Repeated Stratified Search: 100%|██████████| 500/500 [21:33<00:00,  2.59s/it]


✅ 500 次重複分層搜尋 F1 平均值: 0.2623
✅ 95% 信賴區間: [0.1058, 0.3378]
✅ 最佳 F1: 0.3702171664943123
✅ 最佳超參數組合: {'depth': 7, 'learning_rate': 0.23014479072476773, 'l2_leaf_reg': 1.906211260609818, 'bagging_temperature': 0.7129176348350752, 'iterations': 200, 'random_seed': 71}


In [58]:
import pandas as pd
import numpy as np
from imblearn.under_sampling import RandomUnderSampler
from catboost import CatBoostClassifier


# =====================================================
# 5. 最終模型訓練 
# =====================================================

print("🚀 開始對全量訓練資料進行混合採樣 (SMOTE + RUS)...")

# 步驟一 (過採樣): 將正樣本數增加到負樣本數的 10%
over = SMOTE(sampling_strategy=0.2, random_state=42)

# 步驟二 (欠採樣): 將負樣本數減少到正樣本數的 2 倍 (最終比例 1:2)
under = RandomUnderSampler(sampling_strategy=1.0, random_state=42)


X_temp, y_temp = over.fit_resample(X, y)
X_final, y_final = under.fit_resample(X_temp, y_temp)

print(f"✅ 混合採樣完成。最終訓練樣本數: {len(X_final):,}")
print("最終標籤分佈:\n", y_final.value_counts())
# 最終 CatBoost 模型訓練
final_model = CatBoostClassifier(
    **final_params,
    eval_metric="F1",
    task_type="CPU",
    verbose=0,
    thread_count=-1,       
    allow_writing_files=False 
)

print("\n🚀 開始訓練最終 CatBoost 模型 (使用最佳超參數)...")
final_model.fit(X_final, y_final)

# 存模型
model_filename = "final_model_CatBoost_1021_top20_SMOTEandRandomUnderSampler.cbm"
final_model.save_model(model_filename)


print("🎉 最終 CatBoost 模型已完成訓練")
print(f"✅ 模型已儲存為檔案：{model_filename}")

🚀 開始對全量訓練資料進行混合採樣 (SMOTE + RUS)...
✅ 混合採樣完成。最終訓練樣本數: 130,570
最終標籤分佈:
 label
0    65285
1    65285
Name: count, dtype: int64

🚀 開始訓練最終 CatBoost 模型 (使用最佳超參數)...
🎉 最終 CatBoost 模型已完成訓練
✅ 模型已儲存為檔案：final_model_CatBoost_1021_top20_SMOTEandRandomUnderSampler.cbm


In [59]:
# =====================================================
# 6. 產生測試集預測與 submission
# =====================================================
X_test = test_features_final.drop(columns=["acct"])
y_test_pred = final_model.predict(X_test)

submission = pd.DataFrame({
    "acct": test_features_final["acct"],
    "label": y_test_pred
})

print("🎉 已完成預測，submission DataFrame 生成成功")
print(submission.head())

🎉 已完成預測，submission DataFrame 生成成功
                                                acct  label
0  fcf31c5113d3dbd9cb5056045c6a0f213bd8a4fc1bc834...      0
1  e21dfa45e990364194468e501fbfe52ec02a4b71a2e2e8...      0
2  2552e943aaf9caa33183758cd40128ef20a6e6ff16c232...      0
3  71700e7b7c3d40abdfdbcc7afc0752fa8d9bd28b408651...      0
4  c70349fc718ffb88f03f31b5a7fcf65b33dd71dce6fee0...      0


In [60]:
num_ones = (submission["label"] == 1).sum()
num_zeros = (submission["label"] == 0).sum()

print("label=1 筆數:", num_ones)
print("label=0 筆數:", num_zeros)
print("總筆數:", len(submission))

label=1 筆數: 143
label=0 筆數: 4637
總筆數: 4780


In [61]:
file_sub = "submission_template.csv"
submission_tem = pd.read_csv(file_sub)
submission_tem

,acct,label
0,fcf31c5113d3dbd9cb5056045c6a0f213bd8a4fc1bc834...,0
1,e21dfa45e990364194468e501fbfe52ec02a4b71a2e2e8...,0
2,2552e943aaf9caa33183758cd40128ef20a6e6ff16c232...,0
3,71700e7b7c3d40abdfdbcc7afc0752fa8d9bd28b408651...,0
4,c70349fc718ffb88f03f31b5a7fcf65b33dd71dce6fee0...,0
...,...,...
4775,09747f71cc6234a75312f2e77f79b9a51e5145e114e8b0...,0
4776,32e6bf3ca071af026794f4df028f3cdaa43ddd499bac10...,0
4777,e9a6861c68821da506be76419df559efc167ef8a056063...,0
4778,75ca5a3798f3cad4c0e1c1fd0adca48f020b2bda856a8e...,0


In [62]:
# 1. 確認數量是否一樣
print("df_predict:", submission_tem["acct"].nunique())
print("submission:", submission["acct"].nunique())

# 2. 確認 acct 集合是否一樣
same_set = set(submission_tem["acct"]) == set(submission["acct"])
print("兩個集合是否完全相同:", same_set)

# 3. 確認順序是否一樣
same_order = submission_tem["acct"].tolist() == submission["acct"].tolist()
print("兩個 acct 是否順序完全相同:", same_order)

df_predict: 4780
submission: 4780
兩個集合是否完全相同: True
兩個 acct 是否順序完全相同: True


In [63]:
submission.to_csv("submission_CatBoost_1021_top20_SMOTEandRandomUnderSampler.csv", index=False)

# XGBoost （完整特徵＋SMOTE與RandomUnderSampler）

In [66]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import f1_score
from xgboost import XGBClassifier  
from tqdm import tqdm
from sklearn.model_selection import StratifiedShuffleSplit 

# =====================================================
# 1. 拆分資料
# =====================================================
X = train_features_final.drop(columns=["acct","label"])
y = train_features_final["label"]

# 確認維度
print(f"✅ X (特徵集) 尺寸: {X.shape}")
print(f"✅ y (標籤集) 尺寸: {y.shape}")
print(f"✅ X 欄位數 (特徵數): {X.shape[1]}")

# =====================================================
# 2. 🚨 超參數搜尋空間 
# =====================================================
def sample_params(seed):
    rng = np.random.RandomState(seed)
    return {
        "max_depth": rng.randint(4, 10),          
        "learning_rate": rng.uniform(0.01, 0.3),   
        "reg_lambda": rng.uniform(1, 10),         
        "subsample": rng.uniform(0.5, 1.0),       
        "n_estimators": 200,                       
        "random_state": seed                       
    }

# =====================================================
# 重複 Stratified 抽樣 + SMOTE/RandomUnderSampler 訓練 
# =====================================================
n_search_iterations = 500 # 保持原來的 500 次迭代 (可酌情減少以加速測試)
search_results = []
best_params_list = []
best_f1_score = -1
final_params = {}

for i in tqdm(range(n_search_iterations), desc="Repeated Stratified Search (XGB)"):
    
    # 每次迭代執行一次分層抽樣 (70% Train / 30% Validation)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42 + i)
    train_idx, val_idx = next(sss.split(X, y))

    X_train_split, y_train_split = X.iloc[train_idx], y.iloc[train_idx]
    X_val_split, y_val_split = X.iloc[val_idx], y.iloc[val_idx]
    
    # SMOTE + RandomUnderSampler 組合 (流程保持不變)
    # 步驟 1: SMOTE
    over = SMOTE(sampling_strategy=0.1, random_state=42+i)
    
    # 步驟 2: RandomUnderSampler
    under = RandomUnderSampler(sampling_strategy=0.5, random_state=42+i)

    X_temp, y_temp = over.fit_resample(X_train_split, y_train_split)
    X_train_res, y_train_res = under.fit_resample(X_temp, y_temp)

    
    # === 抽一組超參數 ===
    params = sample_params(seed=42+i)

    #  XGBClassifier
    model = XGBClassifier(
        **params,
        # eval_metric="F1", 
        verbosity=0,       
        device="cpu",      
        n_jobs=-1,         
        use_label_encoder=False 
    )

    # 在欠採樣後的訓練集上訓練 
    model.fit(X_train_res, y_train_res)

    # 驗證集上評估 F1 Score
    y_pred = model.predict(X_val_split)
    f1 = f1_score(y_val_split, y_pred)

    search_results.append(f1)
    best_params_list.append(params)
    
    if f1 > best_f1_score:
        best_f1_score = f1
        final_params = params


# =====================================================
# 4. 統計最佳參數 
# =====================================================
mean_f1 = np.mean(search_results)
ci_lower = np.percentile(search_results, 2.5)
ci_upper = np.percentile(search_results, 97.5)

print(f"\n✅ [XGBoost] {n_search_iterations} 次重複分層搜尋 F1 平均值: {mean_f1:.4f}")
print(f"✅ [XGBoost] 95% 信賴區間: [{ci_lower:.4f}, {ci_upper:.4f}]")

print("✅ [XGBoost] 最佳 F1:", best_f1_score)
print("✅ [XGBoost] 最佳超參數組合:", final_params)

✅ X (特徵集) 尺寸: (327429, 104)
✅ y (標籤集) 尺寸: (327429,)
✅ X 欄位數 (特徵數): 104


Repeated Stratified Search (XGB): 100%|██████████| 500/500 [24:20<00:00,  2.92s/it]


✅ [XGBoost] 500 次重複分層搜尋 F1 平均值: 0.3359
✅ [XGBoost] 95% 信賴區間: [0.1524, 0.4052]
✅ [XGBoost] 最佳 F1: 0.42266824085005905
✅ [XGBoost] 最佳超參數組合: {'max_depth': 7, 'learning_rate': 0.2301538621498312, 'reg_lambda': 3.0723718239356907, 'subsample': 0.7820950659633525, 'n_estimators': 200, 'random_state': 444}


In [8]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE 
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier 

# =====================================================
# 5. 最終模型訓練
# =====================================================
final_params={'max_depth': 7, 'learning_rate': 0.2301538621498312, 'reg_lambda': 3.0723718239356907, 'subsample': 0.7820950659633525, 'n_estimators': 200, 'random_state': 444}
print("🚀 開始對全量訓練資料進行混合採樣 (SMOTE + RUS)...")

# SMOTE/RandomUnderSampler
over = SMOTE(sampling_strategy=0.1, random_state=42)
under = RandomUnderSampler(sampling_strategy=0.5, random_state=42)

X_temp, y_temp = over.fit_resample(X, y)
X_final, y_final = under.fit_resample(X_temp, y_temp)

print(f"✅ 混合採樣完成。最終訓練樣本數: {len(X_final):,}")
print("最終標籤分佈:\n", y_final.value_counts())

# 最終 XGBoost 模型訓練
final_model = XGBClassifier(
    **final_params,
    # eval_metric="f1", 
    device="cpu",       
    verbosity=0,        
    n_jobs=-1,           
    use_label_encoder=False 
)

print("\n🚀 開始訓練最終 XGBoost 模型 (使用最佳超參數)...")
final_model.fit(X_final, y_final)

# 儲存模型
model_filename = "final_model_XGBoost_1021_SMOTEandRandomUnderSampler.json"
final_model.save_model(model_filename)


print("🎉 最終 XGBoost 模型已完成訓練")
print(f"✅ 模型已儲存為檔案：{model_filename}")

🚀 開始對全量訓練資料進行混合採樣 (SMOTE + RUS)...
✅ 混合採樣完成。最終訓練樣本數: 97,926
最終標籤分佈:
 label
0    65284
1    32642
Name: count, dtype: int64

🚀 開始訓練最終 XGBoost 模型 (使用最佳超參數)...
🎉 最終 XGBoost 模型已完成訓練
✅ 模型已儲存為檔案：final_model_XGBoost_1021_SMOTEandRandomUnderSampler.json


In [68]:
# =====================================================
# 6. 產生測試集預測與 submission
# =====================================================
X_test = test_features_final.drop(columns=["acct"])
y_test_pred = final_model.predict(X_test)

submission = pd.DataFrame({
    "acct": test_features_final["acct"],
    "label": y_test_pred
})

print("🎉 已完成預測，submission DataFrame 生成成功")
print(submission.head())


🎉 已完成預測，submission DataFrame 生成成功
                                                acct  label
0  fcf31c5113d3dbd9cb5056045c6a0f213bd8a4fc1bc834...      0
1  e21dfa45e990364194468e501fbfe52ec02a4b71a2e2e8...      0
2  2552e943aaf9caa33183758cd40128ef20a6e6ff16c232...      0
3  71700e7b7c3d40abdfdbcc7afc0752fa8d9bd28b408651...      0
4  c70349fc718ffb88f03f31b5a7fcf65b33dd71dce6fee0...      0


In [69]:
num_ones = (submission["label"] == 1).sum()
num_zeros = (submission["label"] == 0).sum()

print("label=1 筆數:", num_ones)
print("label=0 筆數:", num_zeros)
print("總筆數:", len(submission))

label=1 筆數: 95
label=0 筆數: 4685
總筆數: 4780


In [70]:
file_sub = "submission_template.csv"
submission_tem = pd.read_csv(file_sub)
submission_tem

,acct,label
0,fcf31c5113d3dbd9cb5056045c6a0f213bd8a4fc1bc834...,0
1,e21dfa45e990364194468e501fbfe52ec02a4b71a2e2e8...,0
2,2552e943aaf9caa33183758cd40128ef20a6e6ff16c232...,0
3,71700e7b7c3d40abdfdbcc7afc0752fa8d9bd28b408651...,0
4,c70349fc718ffb88f03f31b5a7fcf65b33dd71dce6fee0...,0
...,...,...
4775,09747f71cc6234a75312f2e77f79b9a51e5145e114e8b0...,0
4776,32e6bf3ca071af026794f4df028f3cdaa43ddd499bac10...,0
4777,e9a6861c68821da506be76419df559efc167ef8a056063...,0
4778,75ca5a3798f3cad4c0e1c1fd0adca48f020b2bda856a8e...,0


In [71]:
# 1. 確認數量是否一樣
print("df_predict:", submission_tem["acct"].nunique())
print("submission:", submission["acct"].nunique())

# 2. 確認 acct 集合是否一樣
same_set = set(submission_tem["acct"]) == set(submission["acct"])
print("兩個集合是否完全相同:", same_set)

# 3. 確認順序是否一樣
same_order = submission_tem["acct"].tolist() == submission["acct"].tolist()
print("兩個 acct 是否順序完全相同:", same_order)

df_predict: 4780
submission: 4780
兩個集合是否完全相同: True
兩個 acct 是否順序完全相同: True


In [72]:
submission.to_csv("submission_XGBoost_1021__SMOTEandRandomUnderSampler.csv", index=False)

In [3]:
#特徵篩選

# 1. 指定模型路徑
model_path = "final_model_XGBoost_1021_SMOTEandRandomUnderSampler.json"

# 2. 實例化一個新的 XGBClassifier 物件
loaded_model = XGBClassifier()

# 3. 呼叫 .load_model() 
try:
    loaded_model.load_model(model_path)
    print(f"✅ XGBoost 模型 '{model_path}' 載入成功！")

except Exception as e:
    print(f"❌ 載入模型時發生錯誤：{e}")
    print("👉 請確認您已執行『步驟一』(升級/降級套件) 並且『步驟二』(重啟執行環境)！")

✅ XGBoost 模型 'final_model_XGBoost_1021_SMOTEandRandomUnderSampler.json' 載入成功！


In [11]:
import numpy as np
import pandas as pd # 確保 import pandas

# 確保順序與訓練時一致
feature_names = list(train_features_final.drop(columns=["acct", "label"]).columns)

# 使用 .feature_importances_ 屬性
importances = loaded_model.feature_importances_

# 組成 DataFrame
feat_imp_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

# 顯示前 50 名
print("✅ XGBoost 特徵重要性 (前 50 名):")
display(feat_imp_df.head(50))

✅ XGBoost 特徵重要性 (前 50 名):


,Feature,Importance
28,out_count_7d,0.254995
13,unique_in_partners,0.236779
89,UNK_y,0.079356
47,JPY_x,0.026892
75,04_x,0.024392
17,Rare_Partner_Out_Count,0.019821
74,03_x,0.018805
25,night_in_count,0.016704
27,out_sum_7d,0.016270
80,UNK_x,0.013823


# XGBoost （用前20重要特徵＋SMOTE與RandomUnderSampler）

In [12]:
top20_features = [
    "out_count_7d",
    "unique_in_partners",
    "UNK_y",
    "JPY_x",
    "04_x",
    "Rare_Partner_Out_Count",
    "03_x",
    "night_in_count",
    "out_sum_7d",
    "UNK_x",
    "Max_Hourly_Count",
    "txn_gap_min",
    "in_count",
    "out_mean",
    "out_acct_type_01_ratio",
    "01_x",
    "Rare_Partner_Out_Ratio",
    "05_y",
    "alert_partner_out_cnt",
    "alert_partner_in_cnt"
]

In [13]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
from tqdm import tqdm
from sklearn.model_selection import StratifiedShuffleSplit 

# =====================================================
# 1. 拆分資料 
# =====================================================
# 只保留前 20 特徵作為輸入
X = train_features_final[top20_features].copy()
y = train_features_final["label"].copy()

# 確認維度
print(f"✅ X (特徵集) 尺寸: {X.shape}")
print(f"✅ y (標籤集) 尺寸: {y.shape}")
print(f"✅ X 欄位數 (特徵數): {X.shape[1]}")

# =====================================================
# 2. 超參數搜尋空間 
# =====================================================
def sample_params(seed):
    rng = np.random.RandomState(seed)
    return {
        "max_depth": rng.randint(4, 10),          
        "learning_rate": rng.uniform(0.01, 0.3), 
        "reg_lambda": rng.uniform(1, 10),      
        "subsample": rng.uniform(0.5, 1.0),     
        "n_estimators": 200,                    
        "random_state": seed                   
    }

# =====================================================
# 3. 重複 Stratified 抽樣 + SMOTE + RandomUnderSampler
# =====================================================
n_search_iterations = 500 # 保持原來的 500 次迭代 (可酌情減少以加速測試)
search_results = []
best_params_list = []
best_f1_score = -1
final_params = {}

for i in tqdm(range(n_search_iterations), desc="Repeated Stratified Search (XGB)"):
    
    # 每次迭代執行一次分層抽樣 (70% Train / 30% Validation)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42 + i)
    train_idx, val_idx = next(sss.split(X, y))

    X_train_split, y_train_split = X.iloc[train_idx], y.iloc[train_idx]
    X_val_split, y_val_split = X.iloc[val_idx], y.iloc[val_idx]
    
    # 步驟 1: SMOTE
    over = SMOTE(sampling_strategy=0.1, random_state=42+i)
    
    # 步驟 2: RandomUnderSampler
    under = RandomUnderSampler(sampling_strategy=0.5, random_state=42+i)

    X_temp, y_temp = over.fit_resample(X_train_split, y_train_split)
    X_train_res, y_train_res = under.fit_resample(X_temp, y_temp)

    
    # === 抽一組超參數 ===
    params = sample_params(seed=42+i)

    #  XGBClassifier
    model = XGBClassifier(
        **params,
        # eval_metric="F1", 
        verbosity=0,      
        device="cpu",      
        n_jobs=-1,       
        use_label_encoder=False 
    )

    # 在欠採樣後的訓練集上訓練 
    model.fit(X_train_res, y_train_res)

    # 在驗證集上評估 F1 Score 
    y_pred = model.predict(X_val_split)
    f1 = f1_score(y_val_split, y_pred)

    search_results.append(f1)
    best_params_list.append(params)
    
    if f1 > best_f1_score:
        best_f1_score = f1
        final_params = params


# =====================================================
# 4. 統計最佳參數
# =====================================================
mean_f1 = np.mean(search_results)
ci_lower = np.percentile(search_results, 2.5)
ci_upper = np.percentile(search_results, 97.5)

print(f"\n✅ [XGBoost] {n_search_iterations} 次重複分層搜尋 F1 平均值: {mean_f1:.4f}")
print(f"✅ [XGBoost] 95% 信賴區間: [{ci_lower:.4f}, {ci_upper:.4f}]")

print("✅ [XGBoost] 最佳 F1:", best_f1_score)
print("✅ [XGBoost] 最佳超參數組合:", final_params)

✅ X (特徵集) 尺寸: (327429, 20)
✅ y (標籤集) 尺寸: (327429,)
✅ X 欄位數 (特徵數): 20


Repeated Stratified Search (XGB): 100%|██████████| 500/500 [08:02<00:00,  1.04it/s]


✅ [XGBoost] 500 次重複分層搜尋 F1 平均值: 0.2009
✅ [XGBoost] 95% 信賴區間: [0.1209, 0.2443]
✅ [XGBoost] 最佳 F1: 0.26094137076796037
✅ [XGBoost] 最佳超參數組合: {'max_depth': 8, 'learning_rate': 0.18903818388209617, 'reg_lambda': 7.201934772661773, 'subsample': 0.8774420759049035, 'n_estimators': 200, 'random_state': 448}


In [14]:
# =====================================================
# 5. 最終模型訓練 
# =====================================================
final_params={'max_depth': 7, 'learning_rate': 0.2301538621498312, 'reg_lambda': 3.0723718239356907, 'subsample': 0.7820950659633525, 'n_estimators': 200, 'random_state': 444}
print("🚀 開始對全量訓練資料進行混合採樣 (SMOTE + RUS)...")

# 定義與超參數搜尋時完全相同的採樣策略
over = SMOTE(sampling_strategy=0.1, random_state=42)
under = RandomUnderSampler(sampling_strategy=0.5, random_state=42)

X_temp, y_temp = over.fit_resample(X, y)
X_final, y_final = under.fit_resample(X_temp, y_temp)

print(f"✅ 混合採樣完成。最終訓練樣本數: {len(X_final):,}")
print("最終標籤分佈:\n", y_final.value_counts())

# 2. 最終 XGBoost 模型訓練
final_model = XGBClassifier(
    **final_params,
    # eval_metric="f1", 
    device="cpu",       
    verbosity=0,       
    n_jobs=-1,         
    use_label_encoder=False 
)

print("\n🚀 開始訓練最終 XGBoost 模型 (使用最佳超參數)...")
final_model.fit(X_final, y_final)

# 3. 儲存模型 
model_filename = "final_model_XGBoost_1021_top20_SMOTEandRandomUnderSampler.json"
final_model.save_model(model_filename)

print("🎉 最終 XGBoost 模型已完成訓練")
print(f"✅ 模型已儲存為檔案：{model_filename}")

🚀 開始對全量訓練資料進行混合採樣 (SMOTE + RUS)...
✅ 混合採樣完成。最終訓練樣本數: 97,926
最終標籤分佈:
 label
0    65284
1    32642
Name: count, dtype: int64

🚀 開始訓練最終 XGBoost 模型 (使用最佳超參數)...
🎉 最終 XGBoost 模型已完成訓練
✅ 模型已儲存為檔案：final_model_XGBoost_1021_top20_SMOTEandRandomUnderSampler.json


In [17]:
# =====================================================
# 6. 產生測試集預測與 submission
# =====================================================
X_test = test_features_final[top20_features]
y_test_pred = final_model.predict(X_test)

#產生 submission
submission = pd.DataFrame({
    "acct": test_features_final["acct"],
    "label": y_test_pred
})

print("🎉 已完成預測，submission DataFrame 生成成功")
print(submission.head())

# 5. 統計預測結果 
num_ones = (submission["label"] == 1).sum()
num_zeros = (submission["label"] == 0).sum()

print("\nlabel=1 筆數:", num_ones)
print("label=0 筆數:", num_zeros)
print("總筆數:", len(submission))


🎉 已完成預測，submission DataFrame 生成成功
                                                acct  label
0  fcf31c5113d3dbd9cb5056045c6a0f213bd8a4fc1bc834...      0
1  e21dfa45e990364194468e501fbfe52ec02a4b71a2e2e8...      0
2  2552e943aaf9caa33183758cd40128ef20a6e6ff16c232...      0
3  71700e7b7c3d40abdfdbcc7afc0752fa8d9bd28b408651...      0
4  c70349fc718ffb88f03f31b5a7fcf65b33dd71dce6fee0...      0

label=1 筆數: 137
label=0 筆數: 4643
總筆數: 4780


In [18]:
num_ones = (submission["label"] == 1).sum()
num_zeros = (submission["label"] == 0).sum()

print("label=1 筆數:", num_ones)
print("label=0 筆數:", num_zeros)
print("總筆數:", len(submission))

label=1 筆數: 137
label=0 筆數: 4643
總筆數: 4780


In [19]:
file_sub = "submission_template.csv"
submission_tem = pd.read_csv(file_sub)
submission_tem

,acct,label
0,fcf31c5113d3dbd9cb5056045c6a0f213bd8a4fc1bc834...,0
1,e21dfa45e990364194468e501fbfe52ec02a4b71a2e2e8...,0
2,2552e943aaf9caa33183758cd40128ef20a6e6ff16c232...,0
3,71700e7b7c3d40abdfdbcc7afc0752fa8d9bd28b408651...,0
4,c70349fc718ffb88f03f31b5a7fcf65b33dd71dce6fee0...,0
...,...,...
4775,09747f71cc6234a75312f2e77f79b9a51e5145e114e8b0...,0
4776,32e6bf3ca071af026794f4df028f3cdaa43ddd499bac10...,0
4777,e9a6861c68821da506be76419df559efc167ef8a056063...,0
4778,75ca5a3798f3cad4c0e1c1fd0adca48f020b2bda856a8e...,0


In [20]:
# 1. 確認數量是否一樣
print("df_predict:", submission_tem["acct"].nunique())
print("submission:", submission["acct"].nunique())

# 2. 確認 acct 集合是否一樣
same_set = set(submission_tem["acct"]) == set(submission["acct"])
print("兩個集合是否完全相同:", same_set)

# 3. 確認順序是否一樣
same_order = submission_tem["acct"].tolist() == submission["acct"].tolist()
print("兩個 acct 是否順序完全相同:", same_order)

df_predict: 4780
submission: 4780
兩個集合是否完全相同: True
兩個 acct 是否順序完全相同: True


In [21]:
submission.to_csv("submission_XGBoost_1021_top20_SMOTEandRandomUnderSampler.csv", index=False)

# LGBM（完整特徵＋SMOTE與RandomUnderSampler）

In [6]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import f1_score
from lightgbm import LGBMClassifier 
from tqdm import tqdm
from sklearn.model_selection import StratifiedShuffleSplit

# =====================================================
# 1. 拆分資料 (保持原樣)
# =====================================================
X = train_features_final.drop(columns=["acct","label"])
y = train_features_final["label"]

# 確認維度
print(f"✅ X (特徵集) 尺寸: {X.shape}")
print(f"✅ y (標籤集) 尺寸: {y.shape}")
print(f"✅ X 欄位數 (特徵數): {X.shape[1]}")

# =====================================================
# 2. 超參數搜尋空間 
# =====================================================
def sample_params(seed):
    rng = np.random.RandomState(seed)
    return {
        "max_depth": rng.randint(4, 10),
        "learning_rate": rng.uniform(0.01, 0.3),
        "reg_lambda": rng.uniform(1, 10),         
        "subsample": rng.uniform(0.5, 1.0),      
        "n_estimators": 200,                        
        "random_state": seed
    }

# =====================================================
# 3. 重複 Stratified 抽樣 +SMOTE + RandomUnderSampler
# =====================================================
n_search_iterations = 500 
search_results = []
best_params_list = []
best_f1_score = -1
final_params = {}

for i in tqdm(range(n_search_iterations), desc="Repeated Stratified Search (LGBM)"):
    
    # 分層抽樣
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42 + i)
    train_idx, val_idx = next(sss.split(X, y))

    X_train_split, y_train_split = X.iloc[train_idx], y.iloc[train_idx]
    X_val_split, y_val_split = X.iloc[val_idx], y.iloc[val_idx]
    
    # SMOTE + RandomUnderSampler
    over = SMOTE(sampling_strategy=0.1, random_state=42+i)
    under = RandomUnderSampler(sampling_strategy=0.5, random_state=42+i)

    X_temp, y_temp = over.fit_resample(X_train_split, y_train_split)
    X_train_res, y_train_res = under.fit_resample(X_temp, y_temp)
    
    # === 抽一組超參數 === 
    params = sample_params(seed=42+i)

    # LGBMClassifier
    model = LGBMClassifier(
        **params,
        verbosity=-1,     
        device="cpu",      
        n_jobs=-1,         
    )

    # 在欠採樣後的訓練集上訓練
    model.fit(X_train_res, y_train_res)

    # 驗證集上評估 F1 Score
    y_pred = model.predict(X_val_split)
    f1 = f1_score(y_val_split, y_pred)

    search_results.append(f1)
    best_params_list.append(params)
    
    if f1 > best_f1_score:
        best_f1_score = f1
        final_params = params


# =====================================================
# 4. 統計最佳參數
# =====================================================
mean_f1 = np.mean(search_results)
ci_lower = np.percentile(search_results, 2.5)
ci_upper = np.percentile(search_results, 97.5)

print(f"\n✅ [LGBM] {n_search_iterations} 次重複分層搜尋 F1 平均值: {mean_f1:.4f}")
print(f"✅ [LGBM] 95% 信賴區間: [{ci_lower:.4f}, {ci_upper:.4f}]")

print("✅ [LGBM] 最佳 F1:", best_f1_score)
print("✅ [LGBM] 最佳超參數組合:", final_params)

✅ X (特徵集) 尺寸: (327429, 104)
✅ y (標籤集) 尺寸: (327429,)
✅ X 欄位數 (特徵數): 104


Repeated Stratified Search (LGBM): 100%|██████████| 500/500 [19:59<00:00,  2.40s/it]


✅ [LGBM] 500 次重複分層搜尋 F1 平均值: 0.3416
✅ [LGBM] 95% 信賴區間: [0.1502, 0.4160]
✅ [LGBM] 最佳 F1: 0.432194046306505
✅ [LGBM] 最佳超參數組合: {'max_depth': 8, 'learning_rate': 0.21861783856102168, 'reg_lambda': 5.641834830424265, 'subsample': 0.6354646838481088, 'n_estimators': 200, 'random_state': 416}


In [7]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from lightgbm import LGBMClassifier

# =====================================================
# 5. 最終模型訓練
# =====================================================
final_params={'max_depth': 7, 'learning_rate': 0.2301538621498312, 'reg_lambda': 3.0723718239356907, 'subsample': 0.7820950659633525, 'n_estimators': 200, 'random_state': 444}

print("🚀 開始對全量訓練資料進行混合採樣 (SMOTE + RUS)...")

# 定義採樣策略
over = SMOTE(sampling_strategy=0.1, random_state=42)
under = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
X_temp, y_temp = over.fit_resample(X, y)
X_final, y_final = under.fit_resample(X_temp, y_temp)

print(f"✅ 混合採樣完成。最終訓練樣本數: {len(X_final):,}")
print("最終標籤分佈:\n", y_final.value_counts())

# 2. 最終 LGBM 模型訓練
final_model = LGBMClassifier(
    **final_params,
    device="cpu",      
    verbosity=-1,     
    n_jobs=-1,     
)

print("\n🚀 開始訓練最終 LGBM 模型 (使用最佳超參數)...")
final_model.fit(X_final, y_final)

# 儲存模型
model_filename = "final_model_LGBM_1021_SMOTEandRandomUnderSampler.txt"
final_model.booster_.save_model(model_filename) # 使用 booster_ 內的 save_model 方法


print("🎉 最終 LGBM 模型已完成訓練")
print(f"✅ 模型已儲存為檔案：{model_filename}")


🚀 開始對全量訓練資料進行混合採樣 (SMOTE + RUS)...
✅ 混合採樣完成。最終訓練樣本數: 97,926
最終標籤分佈:
 label
0    65284
1    32642
Name: count, dtype: int64

🚀 開始訓練最終 LGBM 模型 (使用最佳超參數)...
🎉 最終 LGBM 模型已完成訓練
✅ 模型已儲存為檔案：final_model_LGBM_1021_SMOTEandRandomUnderSampler.txt


In [8]:
# =====================================================
# 6. 產生測試集預測與 submission
# =====================================================
X_test = test_features_final.drop(columns=["acct"])
y_test_pred = final_model.predict(X_test)

submission = pd.DataFrame({
    "acct": test_features_final["acct"],
    "label": y_test_pred
})

print("🎉 已完成預測，submission DataFrame 生成成功")
print(submission.head())


🎉 已完成預測，submission DataFrame 生成成功
                                                acct  label
0  fcf31c5113d3dbd9cb5056045c6a0f213bd8a4fc1bc834...      0
1  e21dfa45e990364194468e501fbfe52ec02a4b71a2e2e8...      0
2  2552e943aaf9caa33183758cd40128ef20a6e6ff16c232...      0
3  71700e7b7c3d40abdfdbcc7afc0752fa8d9bd28b408651...      0
4  c70349fc718ffb88f03f31b5a7fcf65b33dd71dce6fee0...      0


In [9]:
num_ones = (submission["label"] == 1).sum()
num_zeros = (submission["label"] == 0).sum()

print("label=1 筆數:", num_ones)
print("label=0 筆數:", num_zeros)
print("總筆數:", len(submission))

label=1 筆數: 98
label=0 筆數: 4682
總筆數: 4780


In [10]:
file_sub = "submission_template.csv"
submission_tem = pd.read_csv(file_sub)
submission_tem

,acct,label
0,fcf31c5113d3dbd9cb5056045c6a0f213bd8a4fc1bc834...,0
1,e21dfa45e990364194468e501fbfe52ec02a4b71a2e2e8...,0
2,2552e943aaf9caa33183758cd40128ef20a6e6ff16c232...,0
3,71700e7b7c3d40abdfdbcc7afc0752fa8d9bd28b408651...,0
4,c70349fc718ffb88f03f31b5a7fcf65b33dd71dce6fee0...,0
...,...,...
4775,09747f71cc6234a75312f2e77f79b9a51e5145e114e8b0...,0
4776,32e6bf3ca071af026794f4df028f3cdaa43ddd499bac10...,0
4777,e9a6861c68821da506be76419df559efc167ef8a056063...,0
4778,75ca5a3798f3cad4c0e1c1fd0adca48f020b2bda856a8e...,0


In [11]:
# 1. 確認數量是否一樣
print("df_predict:", submission_tem["acct"].nunique())
print("submission:", submission["acct"].nunique())

# 2. 確認 acct 集合是否一樣
same_set = set(submission_tem["acct"]) == set(submission["acct"])
print("兩個集合是否完全相同:", same_set)

# 3. 確認順序是否一樣
same_order = submission_tem["acct"].tolist() == submission["acct"].tolist()
print("兩個 acct 是否順序完全相同:", same_order)

df_predict: 4780
submission: 4780
兩個集合是否完全相同: True
兩個 acct 是否順序完全相同: True


In [12]:
submission.to_csv("final_model_LGBM_1021_SMOTEandRandomUnderSampler.csv", index=False)

In [13]:
import lightgbm as lgb
import pandas as pd
import numpy as np

# =====================================================
# 1. 讀取已儲存的 LGBM 模型
# =====================================================

# 1. 定義檔案路徑
model_filename = "final_model_LGBM_1021_SMOTEandRandomUnderSampler.txt"

# 2. 讀取模型
try:
    booster = lgb.Booster(model_file=model_filename)
    print(f"✅ LGBM 模型 '{model_filename}' 載入成功！")
    feature_names = booster.feature_name()
    importances = booster.feature_importance(importance_type='gain')

    feat_imp_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    }).sort_values(by="Importance", ascending=False)

    # 5. 顯示前 50 名
    print("\n✅ LGBM 特徵重要性 (前 50 名，依據 'gain' 排序):")
    display(feat_imp_df.head(50))

except lgb.basic.LightGBMError as e:
    print(f"❌ 載入模型時發生錯誤：{e}")
    print("👉 請確認模型檔案路徑是否正確，以及檔案是否為有效的 LightGBM 模型。")


✅ LGBM 模型 'final_model_LGBM_1021_SMOTEandRandomUnderSampler.txt' 載入成功！

✅ LGBM 特徵重要性 (前 50 名，依據 'gain' 排序):


,Feature,Importance
13,unique_in_partners,146889.474908
89,UNK_y,46973.127376
28,out_count_7d,26466.465002
25,night_in_count,6431.095540
17,Rare_Partner_Out_Count,5624.669195
29,txn_gap_min,4940.163893
31,Max_Hourly_Count,4179.153701
10,in_count,3948.373208
74,03_x,3863.341663
90,net_sum,3516.934360


# LGBM（用前15重要特徵＋SMOTE與RandomUnderSampler）

In [14]:
top15_features_lgbm = [
    "unique_in_partners",
    "UNK_y",
    "out_count_7d",
    "night_in_count",
    "Rare_Partner_Out_Count",
    "txn_gap_min",
    "Max_Hourly_Count",
    "in_count",
    "03_x",
    "net_sum",
    "in_max",
    "out_mean",
    "Rare_Partner_Out_Ratio",
    "UNK_x",
    "03_y"
]

In [15]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import f1_score
from lightgbm import LGBMClassifier 
from tqdm import tqdm
from sklearn.model_selection import StratifiedShuffleSplit

# =====================================================
# 1. 拆分資料 
# =====================================================
X = train_features_final[top15_features_lgbm].copy()
y = train_features_final["label"].copy()

# 確認維度
print(f"✅ X (特徵集) 尺寸: {X.shape}")
print(f"✅ y (標籤集) 尺寸: {y.shape}")
print(f"✅ X 欄位數 (特徵數): {X.shape[1]}")

# =====================================================
# 2.超參數搜尋空間 (LGBM)
# =====================================================
def sample_params(seed):
    rng = np.random.RandomState(seed)
    return {
        "max_depth": rng.randint(4, 10),
        "learning_rate": rng.uniform(0.01, 0.3),
        "reg_lambda": rng.uniform(1, 10),        
        "subsample": rng.uniform(0.5, 1.0),         
        "n_estimators": 200,                        
        "random_state": seed
    }

# =====================================================
# 3. 重複 Stratified 抽樣 + SMOTE + RandomUnderSampler
# =====================================================
n_search_iterations = 500
search_results = []
best_params_list = []
best_f1_score = -1
final_params = {}

for i in tqdm(range(n_search_iterations), desc="Repeated Stratified Search (LGBM)"):
    
    # 分層抽樣 
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42 + i)
    train_idx, val_idx = next(sss.split(X, y))

    X_train_split, y_train_split = X.iloc[train_idx], y.iloc[train_idx]
    X_val_split, y_val_split = X.iloc[val_idx], y.iloc[val_idx]
    
    # SMOTE + RandomUnderSampler
    over = SMOTE(sampling_strategy=0.1, random_state=42+i)
    under = RandomUnderSampler(sampling_strategy=0.5, random_state=42+i)

    X_temp, y_temp = over.fit_resample(X_train_split, y_train_split)
    X_train_res, y_train_res = under.fit_resample(X_temp, y_temp)
    
    # === 抽一組超參數 === 
    params = sample_params(seed=42+i)

    # LGBMClassifier
    model = LGBMClassifier(
        **params,
        verbosity=-1,      
        device="cpu",     
        n_jobs=-1,        
    )

    # 在欠採樣後的訓練集上訓練
    model.fit(X_train_res, y_train_res)

    # 證集 評估 F1 Score
    y_pred = model.predict(X_val_split)
    f1 = f1_score(y_val_split, y_pred)

    search_results.append(f1)
    best_params_list.append(params)
    
    if f1 > best_f1_score:
        best_f1_score = f1
        final_params = params


# =====================================================
# 4. 統計最佳參數 
# =====================================================
mean_f1 = np.mean(search_results)
ci_lower = np.percentile(search_results, 2.5)
ci_upper = np.percentile(search_results, 97.5)

print(f"\n✅ [LGBM] {n_search_iterations} 次重複分層搜尋 F1 平均值: {mean_f1:.4f}")
print(f"✅ [LGBM] 95% 信賴區間: [{ci_lower:.4f}, {ci_upper:.4f}]")

print("✅ [LGBM] 最佳 F1:", best_f1_score)
print("✅ [LGBM] 最佳超參數組合:", final_params)

✅ X (特徵集) 尺寸: (327429, 15)
✅ y (標籤集) 尺寸: (327429,)
✅ X 欄位數 (特徵數): 15


Repeated Stratified Search (LGBM): 100%|██████████| 500/500 [08:08<00:00,  1.02it/s]


✅ [LGBM] 500 次重複分層搜尋 F1 平均值: 0.3089
✅ [LGBM] 95% 信賴區間: [0.1370, 0.3785]
✅ [LGBM] 最佳 F1: 0.4066193853427896
✅ [LGBM] 最佳超參數組合: {'max_depth': 8, 'learning_rate': 0.2034961069732319, 'reg_lambda': 7.443026304658218, 'subsample': 0.539516309743178, 'n_estimators': 200, 'random_state': 203}


In [16]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from lightgbm import LGBMClassifier

# =====================================================
# 5. 最終模型訓練 
# =====================================================
final_params={'max_depth': 7, 'learning_rate': 0.2301538621498312, 'reg_lambda': 3.0723718239356907, 'subsample': 0.7820950659633525, 'n_estimators': 200, 'random_state': 444}

print("🚀 開始對全量訓練資料進行混合採樣 (SMOTE + RUS)...")

# 1. 定義採樣策略 
over = SMOTE(sampling_strategy=0.1, random_state=42)
under = RandomUnderSampler(sampling_strategy=0.5, random_state=42)

X_temp, y_temp = over.fit_resample(X, y)
X_final, y_final = under.fit_resample(X_temp, y_temp)

print(f"✅ 混合採樣完成。最終訓練樣本數: {len(X_final):,}")
print("最終標籤分佈:\n", y_final.value_counts())

# 2.最終 LGBM 模型訓練
final_model = LGBMClassifier(
    **final_params,
    device="cpu",      
    verbosity=-1,    
    n_jobs=-1,     
)

print("\n🚀 開始訓練最終 LGBM 模型 (使用最佳超參數)...")
final_model.fit(X_final, y_final)

# 3. 儲存模型
model_filename = "final_model_LGBM_1021_top15_SMOTEandRandomUnderSampler.txt"
final_model.booster_.save_model(model_filename) # 使用 booster_ 內的 save_model 方法


print("🎉 最終 LGBM 模型已完成訓練")
print(f"✅ 模型已儲存為檔案：{model_filename}")


🚀 開始對全量訓練資料進行混合採樣 (SMOTE + RUS)...
✅ 混合採樣完成。最終訓練樣本數: 97,926
最終標籤分佈:
 label
0    65284
1    32642
Name: count, dtype: int64

🚀 開始訓練最終 LGBM 模型 (使用最佳超參數)...
🎉 最終 LGBM 模型已完成訓練
✅ 模型已儲存為檔案：final_model_LGBM_1021_top15_SMOTEandRandomUnderSampler.txt


In [18]:
# =====================================================
# 6. 產生測試集預測與 submission
# =====================================================
X_test = test_features_final[top15_features_lgbm]
y_test_pred = final_model.predict(X_test)

submission = pd.DataFrame({
    "acct": test_features_final["acct"],
    "label": y_test_pred
})

print("🎉 已完成預測，submission DataFrame 生成成功")
print(submission.head())

🎉 已完成預測，submission DataFrame 生成成功
                                                acct  label
0  fcf31c5113d3dbd9cb5056045c6a0f213bd8a4fc1bc834...      0
1  e21dfa45e990364194468e501fbfe52ec02a4b71a2e2e8...      0
2  2552e943aaf9caa33183758cd40128ef20a6e6ff16c232...      0
3  71700e7b7c3d40abdfdbcc7afc0752fa8d9bd28b408651...      0
4  c70349fc718ffb88f03f31b5a7fcf65b33dd71dce6fee0...      0


In [19]:
num_ones = (submission["label"] == 1).sum()
num_zeros = (submission["label"] == 0).sum()

print("label=1 筆數:", num_ones)
print("label=0 筆數:", num_zeros)
print("總筆數:", len(submission))

label=1 筆數: 116
label=0 筆數: 4664
總筆數: 4780


In [20]:
# 1. 確認數量是否一樣
print("df_predict:", submission_tem["acct"].nunique())
print("submission:", submission["acct"].nunique())

# 2. 確認 acct 集合是否一樣
same_set = set(submission_tem["acct"]) == set(submission["acct"])
print("兩個集合是否完全相同:", same_set)

# 3. 確認順序是否一樣
same_order = submission_tem["acct"].tolist() == submission["acct"].tolist()
print("兩個 acct 是否順序完全相同:", same_order)

df_predict: 4780
submission: 4780
兩個集合是否完全相同: True
兩個 acct 是否順序完全相同: True


In [21]:
submission.to_csv("final_model_LGBM_1021_top15_SMOTEandRandomUnderSampler.csv", index=False)